Masked LSTM

In [ ]:
# Full corrected baseline LSTM code
# Purpose:
#   - report TRAIN / VAL / TEST metrics for the 32 training nodes
#   - report TEST metrics for the 8 held-out nodes
#   - create TRAIN / VAL / TEST scatter plots for the 32 training nodes
#   - create TEST scatter plots for the 8 held-out nodes
#   - create time-series plots with TRAIN / VAL / TEST metric box for training nodes
#   - create time-series plots with TEST metric box for held-out nodes

import os, glob, json, random, gc, warnings, zipfile
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.dates as mdates

import networkx as nx
from geopy.distance import geodesic

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler

try:
    import optuna
except ImportError as e:
    raise ImportError("Install optuna first: pip install optuna") from e

# =========================================================
# 1) CONFIG
# =========================================================
SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("medium")
    except Exception:
        pass

OUT_DIR   = "lstm_cuba_corrected"
PLOTS_DIR = os.path.join(OUT_DIR, "plots")
TS_DIR    = os.path.join(PLOTS_DIR, "timeseries")
SCAT_DIR  = os.path.join(PLOTS_DIR, "scatter")
AN_DIR    = os.path.join(OUT_DIR, "analytics")
NET_DIR   = os.path.join(OUT_DIR, "network")

for d in [OUT_DIR, PLOTS_DIR, TS_DIR, SCAT_DIR, AN_DIR, NET_DIR]:
    os.makedirs(d, exist_ok=True)

SPLIT_JSON          = os.path.join(AN_DIR, "node_split.json")
METRICS_ALL_CSV     = os.path.join(AN_DIR, "per_node_metrics_lstm_all.csv")
TRAIN_METRICS_CSV   = os.path.join(AN_DIR, "per_node_metrics_train_nodes_train_val_test.csv")
HELDOUT_METRICS_CSV = os.path.join(AN_DIR, "per_node_metrics_heldout_test_nodes_test_only.csv")
AGG_TRAIN_CSV       = os.path.join(AN_DIR, "split_aggregates_train_nodes.csv")
AGG_HELDOUT_CSV     = os.path.join(AN_DIR, "split_aggregates_heldout_test_nodes.csv")
BEST_PARAMS         = os.path.join(AN_DIR, "best_params_optuna.json")
BEST_MODEL_PT       = os.path.join(AN_DIR, "best_lstm_optuna.pt")
TRAIN_TABLE_CSV     = os.path.join(AN_DIR, "table_train_nodes_train_val_test.csv")
HELDOUT_TABLE_CSV   = os.path.join(AN_DIR, "table_heldout_nodes_test_only.csv")

SHOW_INLINE = True
SAVE_DPI = 240

mpl.rcParams["figure.dpi"] = 140
mpl.rcParams["savefig.dpi"] = 300
mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 13
mpl.rcParams["xtick.labelsize"] = 11
mpl.rcParams["ytick.labelsize"] = 11
mpl.rcParams["legend.fontsize"] = 11
mpl.rcParams["figure.titlesize"] = 14

TRAIN_START, TRAIN_END = "1979-01-01", "2015-12-31"
VAL_START,   VAL_END   = "2016-01-01", "2019-12-31"
TEST_START,  TEST_END  = "2020-01-01", "2023-12-31"
CUTOFF_MAX            = "2023-12-31"

TRAIN_START_DT = pd.to_datetime(TRAIN_START)
TRAIN_END_DT   = pd.to_datetime(TRAIN_END)
VAL_START_DT   = pd.to_datetime(VAL_START)
VAL_END_DT     = pd.to_datetime(VAL_END)
TEST_START_DT  = pd.to_datetime(TEST_START)
TEST_END_DT    = pd.to_datetime(TEST_END)
CUTOFF_MAX_DT  = pd.to_datetime(CUTOFF_MAX)

R2_THR = 0.60
K_MIN  = 8
ALPHA  = 0.65
L_KM   = 120.0
MIN_W  = 1e-6

LOOKBACK         = 60
N_TRIALS_OPTUNA  = 15
MAX_EPOCHS_TUNE  = 30
PATIENCE_TUNE    = 5
MAX_EPOCHS_FINAL = 60
PATIENCE_FINAL   = 8
BATCH_SIZE       = 64

MC_SAMPLES          = 20
MC_BATCH_SIZE       = 256
MC_PERCENTILE_DTYPE = np.float32

FORECAST_COLOR = "#4c9be8"
MEASURED_COLOR = "orange"
BAND_COLOR     = "#4c9be8"

METRIC_FONTSIZE_TS = 12
METRIC_FONTSIZE_SC = 12
HEADROOM_FACTOR = 1.18

print(f"Seed: {SEED}")
print("Device:", device)

# =========================================================
# 2) HELPERS
# =========================================================
def find_first(patterns):
    for pat in patterns:
        hits = glob.glob(pat, recursive=True)
        if hits:
            return sorted(hits)[0]
    return None

def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b))) if len(a) else np.nan

def mae(a, b):
    return float(mean_absolute_error(a, b)) if len(a) else np.nan

def r2(yt, yp):
    try:
        return float(r2_score(yt, yp))
    except Exception:
        return np.nan

def time_feats(idx: pd.DatetimeIndex) -> np.ndarray:
    m = idx.month.values
    doy = idx.dayofyear.values
    return np.stack(
        [
            np.sin(2*np.pi*m/12.0),
            np.cos(2*np.pi*m/12.0),
            np.sin(2*np.pi*doy/365.0),
            np.cos(2*np.pi*doy/365.0)
        ],
        axis=1
    ).astype(np.float32)

def _maybe_latlon(df: pd.DataFrame):
    lat_col = next((c for c in ["lat", "latitude"] if c in df.columns), None)
    lon_col = next((c for c in ["lon", "longitude"] if c in df.columns), None)
    if lat_col is None or lon_col is None:
        return None
    g = df.dropna(subset=[lat_col, lon_col, "node"])[["node", lat_col, lon_col]].drop_duplicates("node")
    g["node"] = pd.to_numeric(g["node"], errors="coerce").astype("Int64")
    return g.set_index("node").sort_index().rename(columns={lat_col: "lat", lon_col: "lon"})

def metric_box(ax, text_str, loc="upper right", fontsize=12):
    locs = {
        "upper right": (0.98, 0.98, "right", "top"),
        "upper left":  (0.02, 0.98, "left", "top"),
        "lower right": (0.98, 0.02, "right", "bottom"),
        "lower left":  (0.02, 0.02, "left", "bottom"),
    }
    x, y, ha, va = locs[loc]
    ax.text(
        x, y, text_str,
        transform=ax.transAxes,
        ha=ha, va=va,
        fontsize=fontsize,
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="black", alpha=0.88)
    )

def safe_ylim_from_series(*arrays, headroom=1.18):
    vals = []
    for arr in arrays:
        arr = np.asarray(arr, dtype=float)
        arr = arr[np.isfinite(arr)]
        if arr.size:
            vals.append(arr.max())
    if not vals:
        return (0, 1.0)
    ymax = max(vals)
    ymax = max(ymax, 1.0)
    return (0, ymax * headroom)

def set_gcrnn_like_full_axis(ax):
    ax.set_xlim(pd.Timestamp("1979-01-01"), pd.Timestamp("2023-12-31"))
    tick_dates = pd.to_datetime([
        "1979-01-01",
        "1985-06-05",
        "1991-11-09",
        "1998-04-14",
        "2004-09-17",
        "2011-02-21",
        "2017-07-27",
        "2023-12-31"
    ])
    ax.set_xticks(tick_dates)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))

def set_gcrnn_like_test_axis(ax):
    ax.set_xlim(pd.Timestamp("2020-01-01"), pd.Timestamp("2023-12-31"))
    tick_dates = pd.to_datetime([
        "2020-01-01",
        "2020-07-27",
        "2021-02-21",
        "2021-09-17",
        "2022-04-14",
        "2022-11-08",
        "2023-06-05",
        "2023-12-31"
    ])
    ax.set_xticks(tick_dates)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))

# =========================================================
# 3) READ DATA
# =========================================================
DATA_CSV = find_first([
    "./Cuba_Precipitation_with_Nodes (1).csv",
    "./Cuba_Precipitation_with_Nodes.csv",
    "./cuba_precipitation.csv",
    "/kaggle/input/**/Cuba_Precipitation_with_Nodes (1).csv",
    "/kaggle/input/**/Cuba_Precipitation_with_Nodes.csv",
    "/kaggle/input/**/cuba_precipitation*.csv",
    "/content/drive/**/Cuba_Precipitation_with_Nodes (1).csv",
    "/content/drive/**/Cuba_Precipitation_with_Nodes.csv",
    "**/*Cuba*Precipitation*Nodes*.csv",
])

assert DATA_CSV, "Precipitation CSV not found."
print("Using data:", DATA_CSV)

raw = pd.read_csv(DATA_CSV)
raw.columns = [c.lower().strip() for c in raw.columns]

needed = {"node", "time", "precipitation"}
assert needed.issubset(raw.columns), f"CSV must contain {needed}"

raw["time"] = pd.to_datetime(raw["time"], errors="coerce")
raw = raw.dropna(subset=["time"]).copy()
raw = raw.rename(columns={"time": "date", "precipitation": "mm"})
raw["node"] = pd.to_numeric(raw["node"], errors="coerce").astype("Int64")
raw["mm"]   = pd.to_numeric(raw["mm"], errors="coerce").clip(lower=0)
raw = raw[raw["date"] <= CUTOFF_MAX_DT]

wide0 = raw.pivot(index="date", columns="node", values="mm").sort_index()
have_nodes = [int(n) for n in wide0.columns if wide0[n].fillna(0).sum() > 0]
wide = wide0.reindex(columns=have_nodes).fillna(0.0).astype(np.float32)

dates = wide.index
all_nodes = list(wide.columns)
N_NODES = len(all_nodes)

print("Nodes present:", N_NODES)
print("Nodes:", all_nodes)

# =========================================================
# 4) FIXED NODE SPLIT
# =========================================================
TRAIN_FIXED = [1, 3, 4, 5, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
               20, 21, 23, 24, 27, 28, 29, 30, 31, 32, 33, 34, 35, 37, 38, 40]
TEST_FIXED  = [2, 6, 9, 22, 25, 26, 36, 39]

train_nodes = [n for n in TRAIN_FIXED if n in all_nodes]
test_nodes  = [n for n in TEST_FIXED  if n in all_nodes]

with open(SPLIT_JSON, "w") as f:
    json.dump({"seed": int(SEED), "train_nodes": train_nodes, "test_nodes": test_nodes}, f, indent=2)

print("TRAIN:", train_nodes)
print("TEST :", test_nodes)

# Node mask: 1 for the 32 training nodes, 0 for held-out nodes
train_node_mask = np.array([1.0 if n in train_nodes else 0.0 for n in all_nodes], dtype=np.float32)

# =========================================================
# 5) BUILD ADJACENCY (TRAIN ONLY)
# =========================================================
def build_adjacency(wide_df, raw_df, train_start, train_end, r2_thr, k_min, alpha, L_km, eps=1e-9):
    cols = list(wide_df.columns)
    mask = (wide_df.index >= pd.to_datetime(train_start)) & (wide_df.index <= pd.to_datetime(train_end))
    rho = wide_df.loc[mask, cols].corr(method="spearman").astype(float)

    Wcorr = np.square(rho.values)
    Wcorr = np.nan_to_num(Wcorr, nan=0.0)
    Wcorr = 0.5 * (Wcorr + Wcorr.T)
    np.fill_diagonal(Wcorr, 0.0)

    geo = _maybe_latlon(raw_df)
    if geo is not None and set(cols).issubset(set(geo.index)):
        lat = geo.loc[cols, "lat"].values.astype(float)
        lon = geo.loc[cols, "lon"].values.astype(float)
        N = len(cols)
        D = np.zeros((N, N), dtype=float)
        for i in range(N):
            a = (lat[i], lon[i])
            D[i, :] = [geodesic(a, (lat[j], lon[j])).km for j in range(N)]
        Kdist = np.exp(-D / max(L_km, 1e-6))
        np.fill_diagonal(Kdist, 0.0)
        Wsoft = Wcorr * Kdist
    else:
        Wsoft = Wcorr

    Wsoft = np.where(Wcorr >= r2_thr, Wsoft, 0.0)

    keep = np.zeros_like(Wsoft, dtype=bool)
    for i in range(Wsoft.shape[0]):
        idx = np.argsort(Wsoft[i])[::-1]
        top = [j for j in idx if j != i][:k_min]
        keep[i, top] = True
    keep = np.logical_or(keep, keep.T)

    A_nb = np.where(keep, Wsoft, 0.0)
    rowsum = A_nb.sum(axis=1, keepdims=True)
    A_row = np.divide(A_nb, rowsum + eps, where=rowsum > 0)
    A_hat = (alpha * np.eye(A_row.shape[0], dtype=float)) + ((1.0 - alpha) * A_row)

    return A_hat.astype(np.float32), Wsoft.astype(np.float32), cols

A_hat, Wsoft, cols_order = build_adjacency(
    wide, raw, TRAIN_START, TRAIN_END, R2_THR, K_MIN, ALPHA, L_KM
)

wide = wide.reindex(columns=cols_order)
all_nodes = cols_order
N_NODES = len(all_nodes)

adj_csv = os.path.join(NET_DIR, f"adj_weighted_r2_{R2_THR:.2f}_L_{int(L_KM)}_train_1979_2015.csv")
pd.DataFrame(Wsoft, index=cols_order, columns=cols_order).to_csv(adj_csv)
print("Saved adjacency:", adj_csv)

# =========================================================
# 6) NETWORK PLOTS
# =========================================================
coords = _maybe_latlon(raw)

net_png_geo = os.path.join(NET_DIR, "network_softweights_geo.png")
if coords is not None:
    pos_geo = {n: (coords.loc[n, "lon"], coords.loc[n, "lat"]) for n in cols_order if n in coords.index}
    G = nx.Graph()
    G.add_nodes_from(cols_order)

    for i in range(len(cols_order)):
        for j in range(i + 1, len(cols_order)):
            w = float(Wsoft[i, j])
            if w >= MIN_W:
                G.add_edge(cols_order[i], cols_order[j], weight=w)

    fig, ax = plt.subplots(figsize=(13, 5.4))
    ax.set_aspect("equal")

    if G.number_of_edges() > 0:
        wvals = np.array([G[u][v]["weight"] for u, v in G.edges()])
        wmin, wmax = float(wvals.min()), float(wvals.max())
        widths = [0.7 + 4.8 * ((G[u][v]["weight"] - wmin) / (wmax - wmin + 1e-12)) for (u, v) in G.edges()]
        nx.draw_networkx_edges(G, pos_geo, width=widths, edge_color="#808080", alpha=0.70, ax=ax)

    nx.draw_networkx_nodes(G, pos_geo, nodelist=[n for n in cols_order if n in train_nodes],
                           node_color="#1e90ff", edgecolors="black", linewidths=1.1, node_size=650, label="Train", ax=ax)
    nx.draw_networkx_nodes(G, pos_geo, nodelist=[n for n in cols_order if n in test_nodes],
                           node_color="#ffa500", edgecolors="black", linewidths=1.1, node_size=650, label="Test", ax=ax)

    texts = nx.draw_networkx_labels(G, pos_geo, labels={n: str(n) for n in cols_order}, font_size=11, font_weight="bold", ax=ax)
    for t in texts.values():
        t.set_path_effects([pe.withStroke(linewidth=2.2, foreground="white")])

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(True, linestyle="--", alpha=0.30)
    ax.legend(loc="upper right")
    ax.set_title(f"Cuba network — soft weights (ρ²≥{R2_THR:.2f}, L={int(L_KM)} km, Top-K≥{K_MIN})")
    fig.tight_layout()
    fig.savefig(net_png_geo, dpi=SAVE_DPI, bbox_inches="tight")
    if SHOW_INLINE:
        plt.show()
    plt.close(fig)

net_png_circ = os.path.join(NET_DIR, "network_circular.png")
G2 = nx.Graph()
G2.add_nodes_from(cols_order)
for i in range(len(cols_order)):
    for j in range(i + 1, len(cols_order)):
        w = float(Wsoft[i, j])
        if w >= MIN_W:
            G2.add_edge(cols_order[i], cols_order[j], weight=w)

pos_circ = nx.circular_layout(G2)

fig, ax = plt.subplots(figsize=(9.5, 9.5))
if G2.number_of_edges() > 0:
    wvals = np.array([G2[u][v]["weight"] for u, v in G2.edges()])
    wmin, wmax = float(wvals.min()), float(wvals.max())
    widths = [0.7 + 3.6 * ((G2[u][v]["weight"] - wmin) / (wmax - wmin + 1e-12)) for (u, v) in G2.edges()]
    nx.draw_networkx_edges(G2, pos_circ, width=widths, edge_color="#9e9e9e", alpha=0.60, ax=ax)

nx.draw_networkx_nodes(G2, pos_circ, nodelist=[n for n in cols_order if n in train_nodes],
                       node_color="dodgerblue", node_size=650, edgecolors="black", linewidths=1.2, label="Train", ax=ax)
nx.draw_networkx_nodes(G2, pos_circ, nodelist=[n for n in cols_order if n in test_nodes],
                       node_color="orange", node_size=650, edgecolors="black", linewidths=1.2, label="Test", ax=ax)
nx.draw_networkx_labels(G2, pos_circ, labels={n: str(n) for n in cols_order}, font_size=11, font_weight="bold", ax=ax)
ax.set_title(f"Cuba network — circular layout (ρ²≥{R2_THR:.2f}, L={int(L_KM)} km, Top-K≥{K_MIN})")
ax.axis("off")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(net_png_circ, dpi=SAVE_DPI, bbox_inches="tight")
if SHOW_INLINE:
    plt.show()
plt.close(fig)

# =========================================================
# 7) BUILD FEATURES
# =========================================================
log_mm = np.log1p(wide.values.astype(np.float32))

A_feat = Wsoft.copy().astype(np.float32)
np.fill_diagonal(A_feat, 0.0)
row_sums = A_feat.sum(axis=1, keepdims=True)
for i in range(N_NODES):
    if row_sums[i, 0] <= 0:
        A_feat[i, i] = 1.0
row_sums = A_feat.sum(axis=1, keepdims=True)
A_feat = A_feat / row_sums

neighbor_log = log_mm @ A_feat
seas = time_feats(dates)

X_full_raw = np.hstack([log_mm, neighbor_log, seas])
y_full_log = log_mm.copy()

is_train_date = (dates >= TRAIN_START_DT) & (dates <= TRAIN_END_DT)
is_val_date   = (dates >= VAL_START_DT)   & (dates <= VAL_END_DT)
is_test_date  = (dates >= TEST_START_DT)  & (dates <= TEST_END_DT)

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

scaler_X.fit(X_full_raw[is_train_date])
scaler_y.fit(y_full_log[is_train_date])

X_full = scaler_X.transform(X_full_raw).astype(np.float32)
y_full = scaler_y.transform(y_full_log).astype(np.float32)

# =========================================================
# 8) BUILD SEQUENCES
# =========================================================
def build_sequences_for_split(mask_bool, lookback, dates, X_all, y_all):
    mask = mask_bool.astype(bool)
    idx_all = np.arange(len(dates))
    seq_X, seq_y, seq_dates = [], [], []

    for t in idx_all[mask]:
        if t - lookback + 1 < 0:
            continue
        window_idx = np.arange(t - lookback + 1, t + 1)
        if not mask[window_idx].all():
            continue
        seq_X.append(X_all[window_idx, :])
        seq_y.append(y_all[t, :])
        seq_dates.append(dates[t])

    if len(seq_X) == 0:
        return (
            np.empty((0, lookback, X_all.shape[1]), dtype=np.float32),
            np.empty((0, y_all.shape[1]), dtype=np.float32),
            pd.DatetimeIndex([])
        )

    return np.stack(seq_X, axis=0), np.stack(seq_y, axis=0), pd.DatetimeIndex(seq_dates)

X_tr, y_tr, d_tr = build_sequences_for_split(is_train_date, LOOKBACK, dates, X_full, y_full)
X_va, y_va, d_va = build_sequences_for_split(is_val_date,   LOOKBACK, dates, X_full, y_full)
X_te, y_te, d_te = build_sequences_for_split(is_test_date,  LOOKBACK, dates, X_full, y_full)

X_tr_t = torch.tensor(X_tr, dtype=torch.float32, device=device)
y_tr_t = torch.tensor(y_tr, dtype=torch.float32, device=device)
X_va_t = torch.tensor(X_va, dtype=torch.float32, device=device)
y_va_t = torch.tensor(y_va, dtype=torch.float32, device=device)
X_te_t = torch.tensor(X_te, dtype=torch.float32, device=device)
y_te_t = torch.tensor(y_te, dtype=torch.float32, device=device)

# =========================================================
# 9) MODEL
# =========================================================
class SimpleLSTM(nn.Module):
    def __init__(self, n_features_in, n_nodes_out, hidden=64, n_layers=1, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features_in,
            hidden_size=hidden,
            num_layers=n_layers,
            dropout=(dropout if n_layers > 1 else 0.0),
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, n_nodes_out)

    def forward(self, x):
        out, _ = self.lstm(x)
        h_T = out[:, -1, :]
        h_T = self.dropout(h_T)
        y = self.fc(h_T)
        return y

n_features_in = X_full.shape[1]

def train_one_model(model, Xtr, ytr, Xva, yva, lr, wd, max_epochs, patience, batch_size, node_mask_vec, verbose=False):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    best_val = float("inf")
    best_state = None
    pat = 0

    node_mask_t = torch.tensor(node_mask_vec[None, :], dtype=torch.float32, device=device)

    def masked_mse(pred, target):
        mask = node_mask_t.expand(pred.shape[0], -1)
        se = (pred - target) ** 2
        return (se * mask).sum() / mask.sum().clamp_min(1.0)

    def _epoch_loop(X, y, train=True):
        model.train() if train else model.eval()
        N = X.shape[0]
        idx = np.arange(N)
        if train:
            np.random.shuffle(idx)

        total_loss = 0.0
        nb = 0
        with torch.set_grad_enabled(train):
            for i in range(0, N, batch_size):
                bidx = idx[i:i+batch_size]
                xb, yb = X[bidx], y[bidx]
                if train:
                    optimizer.zero_grad(set_to_none=True)
                pred = model(xb)
                loss = masked_mse(pred, yb)
                if train:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                total_loss += float(loss.item())
                nb += 1
        return total_loss / max(1, nb)

    for ep in range(1, max_epochs + 1):
        tr_loss = _epoch_loop(Xtr, ytr, train=True)
        va_loss = _epoch_loop(Xva, yva, train=False)

        if verbose:
            print(f"[Ep {ep:02d}] train masked-MSE={tr_loss:.5f}  val masked-MSE={va_loss:.5f}")

        if va_loss < best_val - 1e-6:
            best_val = va_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                if verbose:
                    print("Early stop.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_val

def objective(trial):
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    random.seed(SEED)

    hidden  = trial.suggest_int("hidden", 32, 160, step=32)
    n_layers = trial.suggest_int("n_layers", 1, 2)
    dropout = trial.suggest_float("dropout", 0.05, 0.30)
    lr      = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    wd      = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)

    model = SimpleLSTM(
        n_features_in=n_features_in,
        n_nodes_out=N_NODES,
        hidden=hidden,
        n_layers=n_layers,
        dropout=dropout
    ).to(device)

    return train_one_model(
        model,
        X_tr_t, y_tr_t,
        X_va_t, y_va_t,
        lr=lr,
        wd=wd,
        max_epochs=MAX_EPOCHS_TUNE,
        patience=PATIENCE_TUNE,
        batch_size=BATCH_SIZE,
        node_mask_vec=train_node_mask,
        verbose=False
    )

print("\nStarting Optuna...")
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=N_TRIALS_OPTUNA, show_progress_bar=False)

with open(BEST_PARAMS, "w") as f:
    json.dump({"best_value": float(study.best_value), "best_params": study.best_trial.params}, f, indent=2)

best_params = study.best_trial.params

final_model = SimpleLSTM(
    n_features_in=n_features_in,
    n_nodes_out=N_NODES,
    hidden=int(best_params["hidden"]),
    n_layers=int(best_params["n_layers"]),
    dropout=float(best_params["dropout"])
).to(device)

_ = train_one_model(
    final_model,
    X_tr_t, y_tr_t,
    X_va_t, y_va_t,
    lr=float(best_params["lr"]),
    wd=float(best_params["weight_decay"]),
    max_epochs=MAX_EPOCHS_FINAL,
    patience=PATIENCE_FINAL,
    batch_size=BATCH_SIZE,
    node_mask_vec=train_node_mask,
    verbose=True
)

torch.save(final_model.state_dict(), BEST_MODEL_PT)

# =========================================================
# 10) STANDARD + OOM-SAFE MC DROPOUT PREDICTIONS
# =========================================================
def inverse_scaled_to_mm(y_scaled, scaler_y):
    y_log = scaler_y.inverse_transform(y_scaled)
    y_mm = np.expm1(y_log)
    return np.clip(y_mm, 0, None)

def model_predict_in_batches(model, X_t, batch_size=256):
    preds = []
    N = X_t.shape[0]
    with torch.no_grad():
        for i in range(0, N, batch_size):
            xb = X_t[i:i+batch_size]
            pb = model(xb).detach().cpu().numpy()
            preds.append(pb)
    return np.vstack(preds)

def mc_dropout_predict_oom_safe(model, X_t, scaler_y, mc_samples=20, batch_size=256):
    model.train()
    sample_preds_mm = []

    for _ in range(mc_samples):
        pred_scaled = model_predict_in_batches(model, X_t, batch_size=batch_size)
        pred_mm = inverse_scaled_to_mm(pred_scaled, scaler_y).astype(MC_PERCENTILE_DTYPE, copy=False)
        sample_preds_mm.append(pred_mm)

        del pred_scaled
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    preds = np.stack(sample_preds_mm, axis=0)
    mean_mm = preds.mean(axis=0)
    p10_mm  = np.percentile(preds, 10, axis=0)
    p90_mm  = np.percentile(preds, 90, axis=0)

    del sample_preds_mm
    del preds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model.eval()
    return mean_mm, p10_mm, p90_mm

y_tr_true_mm = inverse_scaled_to_mm(y_tr, scaler_y)
y_va_true_mm = inverse_scaled_to_mm(y_va, scaler_y)
y_te_true_mm = inverse_scaled_to_mm(y_te, scaler_y)

print("Running OOM-safe MC Dropout...")
y_tr_mc_mean, y_tr_mc_p10, y_tr_mc_p90 = mc_dropout_predict_oom_safe(
    final_model, X_tr_t, scaler_y, mc_samples=MC_SAMPLES, batch_size=MC_BATCH_SIZE
)
y_va_mc_mean, y_va_mc_p10, y_va_mc_p90 = mc_dropout_predict_oom_safe(
    final_model, X_va_t, scaler_y, mc_samples=MC_SAMPLES, batch_size=MC_BATCH_SIZE
)
y_te_mc_mean, y_te_mc_p10, y_te_mc_p90 = mc_dropout_predict_oom_safe(
    final_model, X_te_t, scaler_y, mc_samples=MC_SAMPLES, batch_size=MC_BATCH_SIZE
)

# =========================================================
# 11) METRICS
# =========================================================
def build_metric_rows(node_list, split_name, y_true_arr, y_pred_arr):
    rows = []
    for node in node_list:
        j = all_nodes.index(node)
        rows.append({
            "node": int(node),
            "split": split_name,
            "rmse": rmse(y_true_arr[:, j], y_pred_arr[:, j]),
            "mae": mae(y_true_arr[:, j], y_pred_arr[:, j]),
            "r2":  r2(y_true_arr[:, j], y_pred_arr[:, j]),
        })
    return rows

rows_train_nodes = []
rows_train_nodes += build_metric_rows(train_nodes, "TRAIN", y_tr_true_mm, y_tr_mc_mean)
rows_train_nodes += build_metric_rows(train_nodes, "VAL",   y_va_true_mm, y_va_mc_mean)
rows_train_nodes += build_metric_rows(train_nodes, "TEST",  y_te_true_mm, y_te_mc_mean)

rows_test_nodes = []
rows_test_nodes += build_metric_rows(test_nodes, "TEST", y_te_true_mm, y_te_mc_mean)

met_train_nodes = pd.DataFrame(rows_train_nodes).sort_values(["node", "split"])
met_test_nodes  = pd.DataFrame(rows_test_nodes).sort_values(["node", "split"])

met_all = pd.concat([met_train_nodes, met_test_nodes], axis=0, ignore_index=True)
met_all.to_csv(METRICS_ALL_CSV, index=False)
met_train_nodes.to_csv(TRAIN_METRICS_CSV, index=False)
met_test_nodes.to_csv(HELDOUT_METRICS_CSV, index=False)

def agg_report(df, label):
    part = df[df["split"] == label]
    return {
        "split": label,
        "rmse_mean": float(np.nanmean(part["rmse"])),
        "mae_mean": float(np.nanmean(part["mae"])),
        "r2_mean": float(np.nanmean(part["r2"])),
        "n_nodes": int(part["node"].nunique())
    }

overall_train_nodes = pd.DataFrame([
    agg_report(met_train_nodes, "TRAIN"),
    agg_report(met_train_nodes, "VAL"),
    agg_report(met_train_nodes, "TEST"),
])
overall_heldout_nodes = pd.DataFrame([
    agg_report(met_test_nodes, "TEST"),
])

overall_train_nodes.to_csv(AGG_TRAIN_CSV, index=False)
overall_heldout_nodes.to_csv(AGG_HELDOUT_CSV, index=False)

print("\n=== 32 training nodes: TRAIN / VAL / TEST ===")
print(overall_train_nodes)
print("\n=== 8 held-out nodes: TEST only ===")
print(overall_heldout_nodes)

train_nodes_table = (
    met_train_nodes
    .pivot(index="node", columns="split", values=["rmse", "mae", "r2"])
    .sort_index()
)
train_nodes_table.columns = [f"{metric.upper()}_{split}" for metric, split in train_nodes_table.columns]
train_nodes_table = train_nodes_table.reset_index()
train_nodes_table.to_csv(TRAIN_TABLE_CSV, index=False)

heldout_test_table = (
    met_test_nodes[["node", "rmse", "mae", "r2"]]
    .rename(columns={"rmse": "RMSE_TEST", "mae": "MAE_TEST", "r2": "R2_TEST"})
    .sort_values("node")
)
heldout_test_table.to_csv(HELDOUT_TABLE_CSV, index=False)

# =========================================================
# 12) REBUILD FULL TIMELINE PREDICTIONS
# =========================================================
pred_full_mean = pd.DataFrame(index=dates, columns=all_nodes, data=np.nan, dtype=float)
pred_full_p10  = pd.DataFrame(index=dates, columns=all_nodes, data=np.nan, dtype=float)
pred_full_p90  = pd.DataFrame(index=dates, columns=all_nodes, data=np.nan, dtype=float)

for arr_mean, arr_p10, arr_p90, arr_dates in [
    (y_tr_mc_mean, y_tr_mc_p10, y_tr_mc_p90, d_tr),
    (y_va_mc_mean, y_va_mc_p10, y_va_mc_p90, d_va),
    (y_te_mc_mean, y_te_mc_p10, y_te_mc_p90, d_te),
]:
    for i, dt in enumerate(arr_dates):
        if dt in pred_full_mean.index:
            pred_full_mean.loc[dt, :] = arr_mean[i, :]
            pred_full_p10.loc[dt, :] = arr_p10[i, :]
            pred_full_p90.loc[dt, :] = arr_p90[i, :]

Y_full_mm = wide.copy()

# =========================================================
# 13) TIMESERIES PLOTS
# =========================================================
for node in sorted(train_nodes):
    j = all_nodes.index(node)

    tdates = dates
    meas = Y_full_mm.iloc[:, j].values
    pred = pred_full_mean.iloc[:, j].values
    p10  = pred_full_p10.iloc[:, j].values
    p90  = pred_full_p90.iloc[:, j].values

    fig, ax = plt.subplots(figsize=(13.0, 6.5))

    ax.plot(
        tdates, pred,
        color=FORECAST_COLOR,
        linewidth=1.6,
        label="Forecast (mean, MC dropout)",
        zorder=4
    )
    ax.fill_between(
        tdates, p10, p90,
        color=BAND_COLOR,
        alpha=0.10,
        label="P10–P90",
        zorder=2
    )
    ax.plot(
        tdates, meas,
        color=MEASURED_COLOR,
        linewidth=1.3,
        label="Measured",
        zorder=5
    )

    ax.set_ylim(*safe_ylim_from_series(meas, pred, p90, headroom=HEADROOM_FACTOR))
    set_gcrnn_like_full_axis(ax)

    m_train = met_train_nodes[(met_train_nodes.node == node) & (met_train_nodes.split == "TRAIN")].iloc[0]
    m_val   = met_train_nodes[(met_train_nodes.node == node) & (met_train_nodes.split == "VAL")].iloc[0]
    m_test  = met_train_nodes[(met_train_nodes.node == node) & (met_train_nodes.split == "TEST")].iloc[0]

    txt = (
        f"TRAIN  RMSE={m_train.rmse:.2f}  MAE={m_train.mae:.2f}  R²={m_train.r2:.2f}\n"
        f"VAL    RMSE={m_val.rmse:.2f}  MAE={m_val.mae:.2f}  R²={m_val.r2:.2f}\n"
        f"TEST   RMSE={m_test.rmse:.2f}  MAE={m_test.mae:.2f}  R²={m_test.r2:.2f}"
    )
    metric_box(ax, txt, loc="upper right", fontsize=METRIC_FONTSIZE_TS)

    ax.set_title(f"Node {node} — Timeseries (Train/Val/Test)")
    ax.set_xlabel("Date")
    ax.set_ylabel("mm")
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.30)

    fig.tight_layout()
    out_png = os.path.join(TS_DIR, f"node_{int(node):02d}_timeseries_full.png")
    fig.savefig(out_png, dpi=SAVE_DPI, bbox_inches="tight")
    print("Saved:", out_png)
    if SHOW_INLINE:
        plt.show()
    plt.close(fig)

for node in sorted(test_nodes):
    j = all_nodes.index(node)
    mask = (dates >= TEST_START_DT) & (dates <= TEST_END_DT)

    tdates = dates[mask]
    meas = Y_full_mm.iloc[:, j].values[mask]
    pred = pred_full_mean.iloc[:, j].values[mask]
    p10  = pred_full_p10.iloc[:, j].values[mask]
    p90  = pred_full_p90.iloc[:, j].values[mask]

    fig, ax = plt.subplots(figsize=(12.5, 6.4))

    ax.plot(
        tdates, pred,
        color=FORECAST_COLOR,
        linewidth=1.6,
        label="Forecast (mean, MC dropout)",
        zorder=4
    )
    ax.fill_between(
        tdates, p10, p90,
        color=BAND_COLOR,
        alpha=0.10,
        label="P10–P90",
        zorder=2
    )
    ax.plot(
        tdates, meas,
        color=MEASURED_COLOR,
        linewidth=1.3,
        label="Measured",
        zorder=5
    )

    ax.set_ylim(*safe_ylim_from_series(meas, pred, p90, headroom=HEADROOM_FACTOR))
    set_gcrnn_like_test_axis(ax)

    m_test = met_test_nodes[(met_test_nodes.node == node) & (met_test_nodes.split == "TEST")].iloc[0]
    txt = f"TEST  RMSE={m_test.rmse:.2f}  MAE={m_test.mae:.2f}  R²={m_test.r2:.2f}"
    metric_box(ax, txt, loc="upper right", fontsize=METRIC_FONTSIZE_TS)

    ax.set_title(f"Node {node} — Timeseries (TEST only)")
    ax.set_xlabel("Date")
    ax.set_ylabel("mm")
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.30)

    fig.tight_layout()
    out_png = os.path.join(TS_DIR, f"node_{int(node):02d}_timeseries_test.png")
    fig.savefig(out_png, dpi=SAVE_DPI, bbox_inches="tight")
    print("Saved:", out_png)
    if SHOW_INLINE:
        plt.show()
    plt.close(fig)


# =========================================================
# 14) SCATTER PLOTS — MATCH PARTIAL/GCRNN STYLE
# Metrics lower-right, visible points, no overlap issue
# =========================================================
def scatter_per_split_hex_visible(y_true_mm, y_pred_mm, node, split, out_dir, m_row):
    x = np.log1p(np.clip(y_true_mm, 0, None))
    y = np.log1p(np.clip(y_pred_mm, 0, None))

    finite_mask = np.isfinite(x) & np.isfinite(y)
    x_f = x[finite_mask]
    y_f = y[finite_mask]

    if x_f.size == 0:
        print(f"No valid scatter data for node {node}, {split}")
        return

    fig, ax = plt.subplots(figsize=(5.4, 4.8))

    hb = ax.hexbin(
        x_f, y_f,
        gridsize=60,
        cmap="viridis",
        mincnt=1,
        bins="log",
        zorder=1
    )
    cb = fig.colorbar(hb, ax=ax)
    cb.set_label("Count")

    ax.scatter(
        x_f, y_f,
        s=14,
        alpha=0.55,
        color="purple",
        edgecolors="none",
        label="Daily observations",
        zorder=3
    )

    vmin = min(x_f.min(), y_f.min())
    vmax = max(x_f.max(), y_f.max())
    pad = 0.05 * (vmax - vmin + 1e-9)

    ax.plot(
        [vmin - pad, vmax + pad],
        [vmin - pad, vmax + pad],
        linestyle="--",
        linewidth=1.5,
        color="#1f77b4",
        label="y = x",
        zorder=2
    )

    # Fit line kept to match partially masked/GCRNN style
    if x_f.size > 1:
        a, b = np.polyfit(x_f, y_f, 1)
        xx = np.linspace(vmin - pad, vmax + pad, 100)
        ax.plot(
            xx, a * xx + b,
            color="orange",
            linewidth=2.0,
            label=f"fit: y={a:.2f}x+{b:.2f}",
            zorder=4
        )

    ax.set_xlim(vmin - pad, vmax + pad)
    ax.set_ylim(vmin - pad, vmax + pad)

    ax.set_xlabel("log1p(Actual mm)")
    ax.set_ylabel("log1p(Predicted mm)")
    ax.set_title(f"Node {node} — Scatter ({split}, log-log)")
    ax.grid(True, alpha=0.30)

    ax.legend(loc="upper left", frameon=True)

    # Metrics placed lower-right like partially masked/GCRNN plots
    txt = f"RMSE={m_row.rmse:.2f}\nMAE={m_row.mae:.2f}\nCD={m_row.r2:.2f}"
    ax.text(
        0.96, 0.08, txt,
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=METRIC_FONTSIZE_SC,
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="black", alpha=0.92),
        zorder=10
    )

    fig.tight_layout()
    out_png = os.path.join(out_dir, f"node_{int(node):02d}_scatter_{split.lower()}.png")
    fig.savefig(out_png, dpi=SAVE_DPI, bbox_inches="tight")

    print("Saved scatter:", out_png)

    if SHOW_INLINE:
        plt.show()

    plt.close(fig)


for node in sorted(train_nodes):
    j = all_nodes.index(node)

    m_train = met_train_nodes[(met_train_nodes.node == node) & (met_train_nodes.split == "TRAIN")].iloc[0]
    scatter_per_split_hex_visible(y_tr_true_mm[:, j], y_tr_mc_mean[:, j], node, "TRAIN", SCAT_DIR, m_train)

    m_val = met_train_nodes[(met_train_nodes.node == node) & (met_train_nodes.split == "VAL")].iloc[0]
    scatter_per_split_hex_visible(y_va_true_mm[:, j], y_va_mc_mean[:, j], node, "VAL", SCAT_DIR, m_val)

    m_test = met_train_nodes[(met_train_nodes.node == node) & (met_train_nodes.split == "TEST")].iloc[0]
    scatter_per_split_hex_visible(y_te_true_mm[:, j], y_te_mc_mean[:, j], node, "TEST", SCAT_DIR, m_test)


for node in sorted(test_nodes):
    j = all_nodes.index(node)

    m_test = met_test_nodes[(met_test_nodes.node == node) & (met_test_nodes.split == "TEST")].iloc[0]
    scatter_per_split_hex_visible(y_te_true_mm[:, j], y_te_mc_mean[:, j], node, "TEST", SCAT_DIR, m_test)
# =========================================================
# 15) ZIP OUTPUTS
# =========================================================
zip_name = os.path.join(OUT_DIR, "lstm_metrics_plots_corrected.zip")

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:

    for fp in [
        METRICS_ALL_CSV,
        TRAIN_METRICS_CSV,
        HELDOUT_METRICS_CSV,
        AGG_TRAIN_CSV,
        AGG_HELDOUT_CSV,
        TRAIN_TABLE_CSV,
        HELDOUT_TABLE_CSV,
        BEST_PARAMS,
        SPLIT_JSON,
    ]:
        if fp is not None and os.path.exists(fp):
            zf.write(fp, os.path.join("analytics", os.path.basename(fp)))

    for fp in [adj_csv, net_png_geo, net_png_circ, BEST_MODEL_PT]:
        if fp is not None and os.path.exists(fp):
            zf.write(fp, os.path.join("network", os.path.basename(fp)))

    for base_dir in [SCAT_DIR, TS_DIR]:
        if base_dir is not None and os.path.isdir(base_dir):
            for root, _, files in os.walk(base_dir):
                for f in files:
                    if f.lower().endswith(".png"):
                        full = os.path.join(root, f)
                        rel = os.path.relpath(full, PLOTS_DIR)
                        zf.write(full, os.path.join("plots", rel))

print("\nDone.")
print("Corrected plots folder:", PLOTS_DIR)
print("Analytics folder:", AN_DIR)
print("ZIP:", zip_name)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# =========================================================
# DOWNLOAD / ZIP ALL OUTPUTS
# =========================================================
import os
import zipfile
import gc
import torch

zip_name = os.path.join(OUT_DIR, "lstm_metrics_plots_corrected.zip")

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:

    # Metrics / analytics files
    for fp in [
        METRICS_ALL_CSV,
        TRAIN_METRICS_CSV,
        HELDOUT_METRICS_CSV,
        AGG_TRAIN_CSV,
        AGG_HELDOUT_CSV,
        TRAIN_TABLE_CSV,
        HELDOUT_TABLE_CSV,
        BEST_PARAMS,
        SPLIT_JSON,
    ]:
        if fp is not None and os.path.exists(fp):
            zf.write(fp, os.path.join("analytics", os.path.basename(fp)))

    # Network / model files
    for fp in [adj_csv, net_png_geo, net_png_circ, BEST_MODEL_PT]:
        if fp is not None and os.path.exists(fp):
            zf.write(fp, os.path.join("network", os.path.basename(fp)))

    # Scatter + time-series plots
    for base_dir in [SCAT_DIR, TS_DIR]:
        if base_dir is not None and os.path.isdir(base_dir):
            for root, _, files in os.walk(base_dir):
                for f in files:
                    if f.lower().endswith(".png"):
                        full = os.path.join(root, f)
                        rel = os.path.relpath(full, PLOTS_DIR)
                        zf.write(full, os.path.join("plots", rel))

print("ZIP created:", zip_name)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Partially Masked LSTM

In [ ]:
# Full compromise baseline LSTM code
# Purpose:
#   - fair thesis comparison against the current GCRNN
#   - train on ALL 40 node outputs
#   - validate / early-stop / Optuna-select using ONLY the 32 training nodes
#   - export TRAIN / VAL / TEST metrics for the 32 training nodes
#   - export TEST metrics for the 8 held-out nodes
#   - create TRAIN / VAL / TEST scatter plots for the 32 training nodes
#   - create TEST scatter plots for the 8 held-out nodes
#   - create time-series plots with visible P10–P90 band in GCRNN-like style

import os, glob, json, random, gc, warnings, zipfile
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.dates as mdates

import networkx as nx
from geopy.distance import geodesic

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler

try:
    import optuna
except ImportError as e:
    raise ImportError("Install optuna first: pip install optuna") from e

# =========================================================
# 1) CONFIG
# =========================================================
SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("medium")
    except Exception:
        pass

OUT_DIR   = "lstm_cuba_compromise"
PLOTS_DIR = os.path.join(OUT_DIR, "plots")
TS_DIR    = os.path.join(PLOTS_DIR, "timeseries")
SCAT_DIR  = os.path.join(PLOTS_DIR, "scatter")
AN_DIR    = os.path.join(OUT_DIR, "analytics")
NET_DIR   = os.path.join(OUT_DIR, "network")

for d in [OUT_DIR, PLOTS_DIR, TS_DIR, SCAT_DIR, AN_DIR, NET_DIR]:
    os.makedirs(d, exist_ok=True)

SPLIT_JSON          = os.path.join(AN_DIR, "node_split.json")
METRICS_ALL_CSV     = os.path.join(AN_DIR, "per_node_metrics_lstm_all.csv")
TRAIN_METRICS_CSV   = os.path.join(AN_DIR, "per_node_metrics_train_nodes_train_val_test.csv")
HELDOUT_METRICS_CSV = os.path.join(AN_DIR, "per_node_metrics_heldout_test_nodes_test_only.csv")
AGG_TRAIN_CSV       = os.path.join(AN_DIR, "split_aggregates_train_nodes.csv")
AGG_HELDOUT_CSV     = os.path.join(AN_DIR, "split_aggregates_heldout_test_nodes.csv")
BEST_PARAMS         = os.path.join(AN_DIR, "best_params_optuna.json")
BEST_MODEL_PT       = os.path.join(AN_DIR, "best_lstm_optuna.pt")
TRAIN_TABLE_CSV     = os.path.join(AN_DIR, "table_train_nodes_train_val_test.csv")
HELDOUT_TABLE_CSV   = os.path.join(AN_DIR, "table_heldout_nodes_test_only.csv")

SHOW_INLINE = True
SAVE_DPI = 240

mpl.rcParams["figure.dpi"] = 140
mpl.rcParams["savefig.dpi"] = 300
mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 13
mpl.rcParams["xtick.labelsize"] = 11
mpl.rcParams["ytick.labelsize"] = 11
mpl.rcParams["legend.fontsize"] = 11
mpl.rcParams["figure.titlesize"] = 14

TRAIN_START, TRAIN_END = "1979-01-01", "2015-12-31"
VAL_START,   VAL_END   = "2016-01-01", "2019-12-31"
TEST_START,  TEST_END  = "2020-01-01", "2023-12-31"
CUTOFF_MAX            = "2023-12-31"

TRAIN_START_DT = pd.to_datetime(TRAIN_START)
TRAIN_END_DT   = pd.to_datetime(TRAIN_END)
VAL_START_DT   = pd.to_datetime(VAL_START)
VAL_END_DT     = pd.to_datetime(VAL_END)
TEST_START_DT  = pd.to_datetime(TEST_START)
TEST_END_DT    = pd.to_datetime(TEST_END)
CUTOFF_MAX_DT  = pd.to_datetime(CUTOFF_MAX)

R2_THR = 0.60
K_MIN  = 8
ALPHA  = 0.65
L_KM   = 120.0
MIN_W  = 1e-6

LOOKBACK         = 60
N_TRIALS_OPTUNA  = 15
MAX_EPOCHS_TUNE  = 30
PATIENCE_TUNE    = 5
MAX_EPOCHS_FINAL = 60
PATIENCE_FINAL   = 8
BATCH_SIZE       = 64

MC_SAMPLES          = 20
MC_BATCH_SIZE       = 256
MC_PERCENTILE_DTYPE = np.float32

# Plot styling aligned to the GCRNN style
FORECAST_COLOR = "#1f77b4"
MEASURED_COLOR = "#ff7f0e"
MEAN_COLOR     = "#2ca02c"
BAND_COLOR     = "#9370db"
BAND_ALPHA     = 0.26

METRIC_FONTSIZE_TS = 12
METRIC_FONTSIZE_SC = 12
HEADROOM_FACTOR = 1.35

print(f"Seed: {SEED}")
print("Device:", device)

# =========================================================
# 2) HELPERS
# =========================================================
def find_first(patterns):
    for pat in patterns:
        hits = glob.glob(pat, recursive=True)
        if hits:
            return sorted(hits)[0]
    return None

def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b))) if len(a) else np.nan

def mae(a, b):
    return float(mean_absolute_error(a, b)) if len(a) else np.nan

def r2(yt, yp):
    try:
        return float(r2_score(yt, yp))
    except Exception:
        return np.nan

def time_feats(idx: pd.DatetimeIndex) -> np.ndarray:
    m = idx.month.values
    doy = idx.dayofyear.values
    return np.stack(
        [
            np.sin(2*np.pi*m/12.0),
            np.cos(2*np.pi*m/12.0),
            np.sin(2*np.pi*doy/365.0),
            np.cos(2*np.pi*doy/365.0)
        ],
        axis=1
    ).astype(np.float32)

def _maybe_latlon(df: pd.DataFrame):
    lat_col = next((c for c in ["lat", "latitude"] if c in df.columns), None)
    lon_col = next((c for c in ["lon", "longitude"] if c in df.columns), None)
    if lat_col is None or lon_col is None:
        return None
    g = df.dropna(subset=[lat_col, lon_col, "node"])[["node", lat_col, lon_col]].drop_duplicates("node")
    g["node"] = pd.to_numeric(g["node"], errors="coerce").astype("Int64")
    return g.set_index("node").sort_index().rename(columns={lat_col: "lat", lon_col: "lon"})

def metric_box(ax, text_str, loc="upper right", fontsize=12):
    locs = {
        "upper right": (0.98, 0.98, "right", "top"),
        "upper left":  (0.02, 0.98, "left", "top"),
        "lower right": (0.98, 0.02, "right", "bottom"),
        "lower left":  (0.02, 0.02, "left", "bottom"),
    }
    x, y, ha, va = locs[loc]
    ax.text(
        x, y, text_str,
        transform=ax.transAxes,
        ha=ha, va=va,
        fontsize=fontsize,
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="black", alpha=0.88)
    )

def safe_ylim_from_series(*arrays, headroom=1.35):
    vals = []
    for arr in arrays:
        arr = np.asarray(arr, dtype=float)
        arr = arr[np.isfinite(arr)]
        if arr.size:
            vals.append(arr.max())
    if not vals:
        return (0, 1.0)
    ymax = max(vals)
    ymax = max(ymax, 1.0)
    return (0, ymax * headroom)

def fixed_tick_dates(start_dt, end_dt, n_ticks=8):
    return pd.date_range(start=start_dt, end=end_dt, periods=n_ticks)

# =========================================================
# 3) READ DATA
# =========================================================
DATA_CSV = find_first([
    "./Cuba_Precipitation_with_Nodes (1).csv",
    "./Cuba_Precipitation_with_Nodes.csv",
    "./cuba_precipitation.csv",
    "/kaggle/input/**/Cuba_Precipitation_with_Nodes (1).csv",
    "/kaggle/input/**/Cuba_Precipitation_with_Nodes.csv",
    "/kaggle/input/**/cuba_precipitation*.csv",
    "/content/drive/**/Cuba_Precipitation_with_Nodes (1).csv",
    "/content/drive/**/Cuba_Precipitation_with_Nodes.csv",
    "**/*Cuba*Precipitation*Nodes*.csv",
])

assert DATA_CSV, "Precipitation CSV not found."
print("Using data:", DATA_CSV)

raw = pd.read_csv(DATA_CSV)
raw.columns = [c.lower().strip() for c in raw.columns]

needed = {"node", "time", "precipitation"}
assert needed.issubset(raw.columns), f"CSV must contain {needed}"

raw["time"] = pd.to_datetime(raw["time"], errors="coerce")
raw = raw.dropna(subset=["time"]).copy()
raw = raw.rename(columns={"time": "date", "precipitation": "mm"})
raw["node"] = pd.to_numeric(raw["node"], errors="coerce").astype("Int64")
raw["mm"]   = pd.to_numeric(raw["mm"], errors="coerce").clip(lower=0)
raw = raw[raw["date"] <= CUTOFF_MAX_DT]

wide0 = raw.pivot(index="date", columns="node", values="mm").sort_index()
have_nodes = [int(n) for n in wide0.columns if wide0[n].fillna(0).sum() > 0]
wide = wide0.reindex(columns=have_nodes).fillna(0.0).astype(np.float32)

dates = wide.index
all_nodes = list(wide.columns)
N_NODES = len(all_nodes)

print("Nodes present:", N_NODES)
print("Nodes:", all_nodes)

# =========================================================
# 4) FIXED NODE SPLIT
# =========================================================
TRAIN_FIXED = [1, 3, 4, 5, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
               20, 21, 23, 24, 27, 28, 29, 30, 31, 32, 33, 34, 35, 37, 38, 40]
TEST_FIXED  = [2, 6, 9, 22, 25, 26, 36, 39]

train_nodes = [n for n in TRAIN_FIXED if n in all_nodes]
test_nodes  = [n for n in TEST_FIXED  if n in all_nodes]

with open(SPLIT_JSON, "w") as f:
    json.dump({"seed": int(SEED), "train_nodes": train_nodes, "test_nodes": test_nodes}, f, indent=2)

print("TRAIN:", train_nodes)
print("TEST :", test_nodes)

train_node_mask = np.array([1.0 if n in train_nodes else 0.0 for n in all_nodes], dtype=np.float32)

# =========================================================
# 5) BUILD ADJACENCY (TRAIN ONLY) -- FOR MATCHED NETWORK PLOTS/FEATURES
# =========================================================
def build_adjacency(wide_df, raw_df, train_start, train_end, r2_thr, k_min, alpha, L_km, eps=1e-9):
    cols = list(wide_df.columns)
    mask = (wide_df.index >= pd.to_datetime(train_start)) & (wide_df.index <= pd.to_datetime(train_end))
    rho = wide_df.loc[mask, cols].corr(method="spearman").astype(float)

    Wcorr = np.square(rho.values)
    Wcorr = np.nan_to_num(Wcorr, nan=0.0)
    Wcorr = 0.5 * (Wcorr + Wcorr.T)
    np.fill_diagonal(Wcorr, 0.0)

    geo = _maybe_latlon(raw_df)
    if geo is not None and set(cols).issubset(set(geo.index)):
        lat = geo.loc[cols, "lat"].values.astype(float)
        lon = geo.loc[cols, "lon"].values.astype(float)
        N = len(cols)
        D = np.zeros((N, N), dtype=float)
        for i in range(N):
            a = (lat[i], lon[i])
            D[i, :] = [geodesic(a, (lat[j], lon[j])).km for j in range(N)]
        Kdist = np.exp(-D / max(L_km, 1e-6))
        np.fill_diagonal(Kdist, 0.0)
        Wsoft = Wcorr * Kdist
    else:
        Wsoft = Wcorr

    Wsoft = np.where(Wcorr >= r2_thr, Wsoft, 0.0)

    keep = np.zeros_like(Wsoft, dtype=bool)
    for i in range(Wsoft.shape[0]):
        idx = np.argsort(Wsoft[i])[::-1]
        top = [j for j in idx if j != i][:k_min]
        keep[i, top] = True
    keep = np.logical_or(keep, keep.T)

    A_nb = np.where(keep, Wsoft, 0.0)
    rowsum = A_nb.sum(axis=1, keepdims=True)
    A_row = np.divide(A_nb, rowsum + eps, where=rowsum > 0)
    A_hat = (alpha * np.eye(A_row.shape[0], dtype=float)) + ((1.0 - alpha) * A_row)

    return A_hat.astype(np.float32), Wsoft.astype(np.float32), cols

A_hat, Wsoft, cols_order = build_adjacency(
    wide, raw, TRAIN_START, TRAIN_END, R2_THR, K_MIN, ALPHA, L_KM
)

wide = wide.reindex(columns=cols_order)
all_nodes = cols_order
N_NODES = len(all_nodes)
train_node_mask = np.array([1.0 if n in train_nodes else 0.0 for n in all_nodes], dtype=np.float32)

adj_csv = os.path.join(NET_DIR, f"adj_weighted_r2_{R2_THR:.2f}_L_{int(L_KM)}_train_1979_2015.csv")
pd.DataFrame(Wsoft, index=cols_order, columns=cols_order).to_csv(adj_csv)
print("Saved adjacency:", adj_csv)

# =========================================================
# 6) NETWORK PLOTS
# =========================================================
coords = _maybe_latlon(raw)

net_png_geo = os.path.join(NET_DIR, "network_softweights_geo.png")
if coords is not None:
    pos_geo = {n: (coords.loc[n, "lon"], coords.loc[n, "lat"]) for n in cols_order if n in coords.index}
    G = nx.Graph()
    G.add_nodes_from(cols_order)

    for i in range(len(cols_order)):
        for j in range(i + 1, len(cols_order)):
            w = float(Wsoft[i, j])
            if w >= MIN_W:
                G.add_edge(cols_order[i], cols_order[j], weight=w)

    fig, ax = plt.subplots(figsize=(13, 5.4))
    ax.set_aspect("equal")

    if G.number_of_edges() > 0:
        wvals = np.array([G[u][v]["weight"] for u, v in G.edges()])
        wmin, wmax = float(wvals.min()), float(wvals.max())
        widths = [0.7 + 4.8 * ((G[u][v]["weight"] - wmin) / (wmax - wmin + 1e-12)) for (u, v) in G.edges()]
        nx.draw_networkx_edges(G, pos_geo, width=widths, edge_color="#808080", alpha=0.70, ax=ax)

    nx.draw_networkx_nodes(G, pos_geo, nodelist=[n for n in cols_order if n in train_nodes],
                           node_color="#1e90ff", edgecolors="black", linewidths=1.1, node_size=650, label="Train", ax=ax)
    nx.draw_networkx_nodes(G, pos_geo, nodelist=[n for n in cols_order if n in test_nodes],
                           node_color="#ffa500", edgecolors="black", linewidths=1.1, node_size=650, label="Test", ax=ax)

    texts = nx.draw_networkx_labels(G, pos_geo, labels={n: str(n) for n in cols_order}, font_size=11, font_weight="bold", ax=ax)
    for t in texts.values():
        t.set_path_effects([pe.withStroke(linewidth=2.2, foreground="white")])

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(True, linestyle="--", alpha=0.30)
    ax.legend(loc="upper right")
    ax.set_title(f"Cuba network — soft weights (ρ²≥{R2_THR:.2f}, L={int(L_KM)} km, Top-K≥{K_MIN})")
    fig.tight_layout()
    fig.savefig(net_png_geo, dpi=SAVE_DPI, bbox_inches="tight")
    if SHOW_INLINE:
        plt.show()
    plt.close(fig)

net_png_circ = os.path.join(NET_DIR, "network_circular.png")
G2 = nx.Graph()
G2.add_nodes_from(cols_order)
for i in range(len(cols_order)):
    for j in range(i + 1, len(cols_order)):
        w = float(Wsoft[i, j])
        if w >= MIN_W:
            G2.add_edge(cols_order[i], cols_order[j], weight=w)

pos_circ = nx.circular_layout(G2)

fig, ax = plt.subplots(figsize=(9.5, 9.5))
if G2.number_of_edges() > 0:
    wvals = np.array([G2[u][v]["weight"] for u, v in G2.edges()])
    wmin, wmax = float(wvals.min()), float(wvals.max())
    widths = [0.7 + 3.6 * ((G2[u][v]["weight"] - wmin) / (wmax - wmin + 1e-12)) for (u, v) in G2.edges()]
    nx.draw_networkx_edges(G2, pos_circ, width=widths, edge_color="#9e9e9e", alpha=0.60, ax=ax)

nx.draw_networkx_nodes(G2, pos_circ, nodelist=[n for n in cols_order if n in train_nodes],
                       node_color="dodgerblue", node_size=650, edgecolors="black", linewidths=1.2, label="Train", ax=ax)
nx.draw_networkx_nodes(G2, pos_circ, nodelist=[n for n in cols_order if n in test_nodes],
                       node_color="orange", node_size=650, edgecolors="black", linewidths=1.2, label="Test", ax=ax)
nx.draw_networkx_labels(G2, pos_circ, labels={n: str(n) for n in cols_order}, font_size=11, font_weight="bold", ax=ax)
ax.set_title(f"Cuba network — circular layout (ρ²≥{R2_THR:.2f}, L={int(L_KM)} km, Top-K≥{K_MIN})")
ax.axis("off")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(net_png_circ, dpi=SAVE_DPI, bbox_inches="tight")
if SHOW_INLINE:
    plt.show()
plt.close(fig)

# =========================================================
# 7) BUILD FEATURES
# =========================================================
log_mm = np.log1p(wide.values.astype(np.float32))

# Keep neighbor feature so the baseline uses the same input summary as your recent file style
A_feat = Wsoft.copy().astype(np.float32)
np.fill_diagonal(A_feat, 0.0)
row_sums = A_feat.sum(axis=1, keepdims=True)
for i in range(N_NODES):
    if row_sums[i, 0] <= 0:
        A_feat[i, i] = 1.0
row_sums = A_feat.sum(axis=1, keepdims=True)
A_feat = A_feat / row_sums

neighbor_log = log_mm @ A_feat
seas = time_feats(dates)

X_full_raw = np.hstack([log_mm, neighbor_log, seas])
y_full_log = log_mm.copy()

is_train_date = (dates >= TRAIN_START_DT) & (dates <= TRAIN_END_DT)
is_val_date   = (dates >= VAL_START_DT)   & (dates <= VAL_END_DT)
is_test_date  = (dates >= TEST_START_DT)  & (dates <= TEST_END_DT)

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

scaler_X.fit(X_full_raw[is_train_date])
scaler_y.fit(y_full_log[is_train_date])

X_full = scaler_X.transform(X_full_raw).astype(np.float32)
y_full = scaler_y.transform(y_full_log).astype(np.float32)

# =========================================================
# 8) BUILD SEQUENCES
# =========================================================
def build_sequences_for_split(mask_bool, lookback, dates, X_all, y_all):
    mask = mask_bool.astype(bool)
    idx_all = np.arange(len(dates))
    seq_X, seq_y, seq_dates = [], [], []

    for t in idx_all[mask]:
        if t - lookback + 1 < 0:
            continue
        window_idx = np.arange(t - lookback + 1, t + 1)
        if not mask[window_idx].all():
            continue
        seq_X.append(X_all[window_idx, :])
        seq_y.append(y_all[t, :])
        seq_dates.append(dates[t])

    if len(seq_X) == 0:
        return (
            np.empty((0, lookback, X_all.shape[1]), dtype=np.float32),
            np.empty((0, y_all.shape[1]), dtype=np.float32),
            pd.DatetimeIndex([])
        )

    return np.stack(seq_X, axis=0), np.stack(seq_y, axis=0), pd.DatetimeIndex(seq_dates)

X_tr, y_tr, d_tr = build_sequences_for_split(is_train_date, LOOKBACK, dates, X_full, y_full)
X_va, y_va, d_va = build_sequences_for_split(is_val_date,   LOOKBACK, dates, X_full, y_full)
X_te, y_te, d_te = build_sequences_for_split(is_test_date,  LOOKBACK, dates, X_full, y_full)

X_tr_t = torch.tensor(X_tr, dtype=torch.float32, device=device)
y_tr_t = torch.tensor(y_tr, dtype=torch.float32, device=device)
X_va_t = torch.tensor(X_va, dtype=torch.float32, device=device)
y_va_t = torch.tensor(y_va, dtype=torch.float32, device=device)
X_te_t = torch.tensor(X_te, dtype=torch.float32, device=device)
y_te_t = torch.tensor(y_te, dtype=torch.float32, device=device)

# =========================================================
# 9) MODEL
# =========================================================
class SimpleLSTM(nn.Module):
    def __init__(self, n_features_in, n_nodes_out, hidden=64, n_layers=1, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features_in,
            hidden_size=hidden,
            num_layers=n_layers,
            dropout=(dropout if n_layers > 1 else 0.0),
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, n_nodes_out)

    def forward(self, x):
        out, _ = self.lstm(x)
        h_T = out[:, -1, :]
        h_T = self.dropout(h_T)
        y = self.fc(h_T)
        return y

n_features_in = X_full.shape[1]

# =========================================================
# 10) TRAINING LOGIC
#   compromise version:
#   - training loss: ALL nodes
#   - validation loss: ONLY training nodes
# =========================================================
def train_one_model(model, Xtr, ytr, Xva, yva, lr, wd, max_epochs, patience, batch_size, node_mask_vec, verbose=False):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    best_val = float("inf")
    best_state = None
    pat = 0

    node_mask_t = torch.tensor(node_mask_vec[None, :], dtype=torch.float32, device=device)

    def full_mse(pred, target):
        return ((pred - target) ** 2).mean()

    def masked_val_mse(pred, target):
        mask = node_mask_t.expand(pred.shape[0], -1)
        se = (pred - target) ** 2
        return (se * mask).sum() / mask.sum().clamp_min(1.0)

    def _epoch_loop(X, y, train=True):
        model.train() if train else model.eval()
        N = X.shape[0]
        idx = np.arange(N)
        if train:
            np.random.shuffle(idx)

        total_loss = 0.0
        nb = 0

        with torch.set_grad_enabled(train):
            for i in range(0, N, batch_size):
                bidx = idx[i:i+batch_size]
                xb, yb = X[bidx], y[bidx]

                if train:
                    optimizer.zero_grad(set_to_none=True)

                pred = model(xb)

                if train:
                    # ALL-node training loss
                    loss = full_mse(pred, yb)
                else:
                    # masked validation on 32 training nodes only
                    loss = masked_val_mse(pred, yb)

                if train:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()

                total_loss += float(loss.item())
                nb += 1

        return total_loss / max(1, nb)

    for ep in range(1, max_epochs + 1):
        tr_loss = _epoch_loop(Xtr, ytr, train=True)
        va_loss = _epoch_loop(Xva, yva, train=False)

        if verbose:
            print(f"[Ep {ep:02d}] train all-node MSE={tr_loss:.5f}  val masked-MSE={va_loss:.5f}")

        if va_loss < best_val - 1e-6:
            best_val = va_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                if verbose:
                    print("Early stop.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_val

def objective(trial):
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    random.seed(SEED)

    hidden   = trial.suggest_int("hidden", 32, 160, step=32)
    n_layers = trial.suggest_int("n_layers", 1, 2)
    dropout  = trial.suggest_float("dropout", 0.05, 0.30)
    lr       = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    wd       = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)

    model = SimpleLSTM(
        n_features_in=n_features_in,
        n_nodes_out=N_NODES,
        hidden=hidden,
        n_layers=n_layers,
        dropout=dropout
    ).to(device)

    return train_one_model(
        model,
        X_tr_t, y_tr_t,
        X_va_t, y_va_t,
        lr=lr,
        wd=wd,
        max_epochs=MAX_EPOCHS_TUNE,
        patience=PATIENCE_TUNE,
        batch_size=BATCH_SIZE,
        node_mask_vec=train_node_mask,
        verbose=False
    )

print("\nStarting Optuna...")
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=N_TRIALS_OPTUNA, show_progress_bar=False)

with open(BEST_PARAMS, "w") as f:
    json.dump({"best_value": float(study.best_value), "best_params": study.best_trial.params}, f, indent=2)

best_params = study.best_trial.params

final_model = SimpleLSTM(
    n_features_in=n_features_in,
    n_nodes_out=N_NODES,
    hidden=int(best_params["hidden"]),
    n_layers=int(best_params["n_layers"]),
    dropout=float(best_params["dropout"])
).to(device)

_ = train_one_model(
    final_model,
    X_tr_t, y_tr_t,
    X_va_t, y_va_t,
    lr=float(best_params["lr"]),
    wd=float(best_params["weight_decay"]),
    max_epochs=MAX_EPOCHS_FINAL,
    patience=PATIENCE_FINAL,
    batch_size=BATCH_SIZE,
    node_mask_vec=train_node_mask,
    verbose=True
)

torch.save(final_model.state_dict(), BEST_MODEL_PT)

# =========================================================
# 11) MC DROPOUT PREDICTIONS
# =========================================================
def inverse_scaled_to_mm(y_scaled, scaler_y):
    y_log = scaler_y.inverse_transform(y_scaled)
    y_mm = np.expm1(y_log)
    return np.clip(y_mm, 0, None)

def model_predict_in_batches(model, X_t, batch_size=256):
    preds = []
    N = X_t.shape[0]
    with torch.no_grad():
        for i in range(0, N, batch_size):
            xb = X_t[i:i+batch_size]
            pb = model(xb).detach().cpu().numpy()
            preds.append(pb)
    return np.vstack(preds)

def mc_dropout_predict_oom_safe(model, X_t, scaler_y, mc_samples=20, batch_size=256):
    model.train()
    sample_preds_mm = []

    for _ in range(mc_samples):
        pred_scaled = model_predict_in_batches(model, X_t, batch_size=batch_size)
        pred_mm = inverse_scaled_to_mm(pred_scaled, scaler_y).astype(MC_PERCENTILE_DTYPE, copy=False)
        sample_preds_mm.append(pred_mm)

        del pred_scaled
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    preds = np.stack(sample_preds_mm, axis=0)
    mean_mm = preds.mean(axis=0)
    p10_mm  = np.percentile(preds, 10, axis=0)
    p90_mm  = np.percentile(preds, 90, axis=0)

    del sample_preds_mm
    del preds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model.eval()
    return mean_mm, p10_mm, p90_mm

y_tr_true_mm = inverse_scaled_to_mm(y_tr, scaler_y)
y_va_true_mm = inverse_scaled_to_mm(y_va, scaler_y)
y_te_true_mm = inverse_scaled_to_mm(y_te, scaler_y)

print("Running OOM-safe MC Dropout...")
y_tr_mc_mean, y_tr_mc_p10, y_tr_mc_p90 = mc_dropout_predict_oom_safe(
    final_model, X_tr_t, scaler_y, mc_samples=MC_SAMPLES, batch_size=MC_BATCH_SIZE
)
y_va_mc_mean, y_va_mc_p10, y_va_mc_p90 = mc_dropout_predict_oom_safe(
    final_model, X_va_t, scaler_y, mc_samples=MC_SAMPLES, batch_size=MC_BATCH_SIZE
)
y_te_mc_mean, y_te_mc_p10, y_te_mc_p90 = mc_dropout_predict_oom_safe(
    final_model, X_te_t, scaler_y, mc_samples=MC_SAMPLES, batch_size=MC_BATCH_SIZE
)

# =========================================================
# 12) METRICS
# =========================================================
def build_metric_rows(node_list, split_name, y_true_arr, y_pred_arr):
    rows = []
    for node in node_list:
        j = all_nodes.index(node)
        rows.append({
            "node": int(node),
            "split": split_name,
            "rmse": rmse(y_true_arr[:, j], y_pred_arr[:, j]),
            "mae": mae(y_true_arr[:, j], y_pred_arr[:, j]),
            "r2":  r2(y_true_arr[:, j], y_pred_arr[:, j]),
        })
    return rows

rows_train_nodes = []
rows_train_nodes += build_metric_rows(train_nodes, "TRAIN", y_tr_true_mm, y_tr_mc_mean)
rows_train_nodes += build_metric_rows(train_nodes, "VAL",   y_va_true_mm, y_va_mc_mean)
rows_train_nodes += build_metric_rows(train_nodes, "TEST",  y_te_true_mm, y_te_mc_mean)

rows_test_nodes = []
rows_test_nodes += build_metric_rows(test_nodes, "TEST", y_te_true_mm, y_te_mc_mean)

met_train_nodes = pd.DataFrame(rows_train_nodes).sort_values(["node", "split"])
met_test_nodes  = pd.DataFrame(rows_test_nodes).sort_values(["node", "split"])

met_all = pd.concat([met_train_nodes, met_test_nodes], axis=0, ignore_index=True)
met_all.to_csv(METRICS_ALL_CSV, index=False)
met_train_nodes.to_csv(TRAIN_METRICS_CSV, index=False)
met_test_nodes.to_csv(HELDOUT_METRICS_CSV, index=False)

def agg_report(df, label):
    part = df[df["split"] == label]
    return {
        "split": label,
        "rmse_mean": float(np.nanmean(part["rmse"])),
        "mae_mean": float(np.nanmean(part["mae"])),
        "r2_mean": float(np.nanmean(part["r2"])),
        "n_nodes": int(part["node"].nunique())
    }

overall_train_nodes = pd.DataFrame([
    agg_report(met_train_nodes, "TRAIN"),
    agg_report(met_train_nodes, "VAL"),
    agg_report(met_train_nodes, "TEST"),
])
overall_heldout_nodes = pd.DataFrame([
    agg_report(met_test_nodes, "TEST"),
])

overall_train_nodes.to_csv(AGG_TRAIN_CSV, index=False)
overall_heldout_nodes.to_csv(AGG_HELDOUT_CSV, index=False)

print("\n=== 32 training nodes: TRAIN / VAL / TEST ===")
print(overall_train_nodes)
print("\n=== 8 held-out nodes: TEST only ===")
print(overall_heldout_nodes)

train_nodes_table = (
    met_train_nodes
    .pivot(index="node", columns="split", values=["rmse", "mae", "r2"])
    .sort_index()
)
train_nodes_table.columns = [f"{metric.upper()}_{split}" for metric, split in train_nodes_table.columns]
train_nodes_table = train_nodes_table.reset_index()
train_nodes_table.to_csv(TRAIN_TABLE_CSV, index=False)

heldout_test_table = (
    met_test_nodes[["node", "rmse", "mae", "r2"]]
    .rename(columns={"rmse": "RMSE_TEST", "mae": "MAE_TEST", "r2": "R2_TEST"})
    .sort_values("node")
)
heldout_test_table.to_csv(HELDOUT_TABLE_CSV, index=False)

# =========================================================
# 13) REBUILD FULL TIMELINE PREDICTIONS
# =========================================================
pred_full_mean = pd.DataFrame(index=dates, columns=all_nodes, data=np.nan, dtype=float)
pred_full_p10  = pd.DataFrame(index=dates, columns=all_nodes, data=np.nan, dtype=float)
pred_full_p90  = pd.DataFrame(index=dates, columns=all_nodes, data=np.nan, dtype=float)

for arr_mean, arr_p10, arr_p90, arr_dates in [
    (y_tr_mc_mean, y_tr_mc_p10, y_tr_mc_p90, d_tr),
    (y_va_mc_mean, y_va_mc_p10, y_va_mc_p90, d_va),
    (y_te_mc_mean, y_te_mc_p10, y_te_mc_p90, d_te),
]:
    for i, dt in enumerate(arr_dates):
        if dt in pred_full_mean.index:
            pred_full_mean.loc[dt, :] = arr_mean[i, :]
            pred_full_p10.loc[dt, :] = arr_p10[i, :]
            pred_full_p90.loc[dt, :] = arr_p90[i, :]

Y_full_mm = wide.copy()

# =========================================================
# 14) TIMESERIES PLOTS
# =========================================================
for node in sorted(train_nodes):
    j = all_nodes.index(node)

    tdates = dates
    meas = Y_full_mm.iloc[:, j].values
    pred = pred_full_mean.iloc[:, j].values
    p10  = pred_full_p10.iloc[:, j].values
    p90  = pred_full_p90.iloc[:, j].values

    fig, ax = plt.subplots(figsize=(12.0, 6.5))

    ax.plot(
        tdates, pred,
        color=FORECAST_COLOR,
        linewidth=1.8,
        label="Forecast (mean, MC dropout)",
        zorder=4
    )
    ax.fill_between(
        tdates, p10, p90,
        color=BAND_COLOR,
        alpha=BAND_ALPHA,
        label="P10–P90 (MC)",
        zorder=1
    )
    ax.plot(
        tdates, meas,
        color=MEASURED_COLOR,
        linewidth=1.6,
        label="Measured",
        zorder=6,
        path_effects=[pe.Stroke(linewidth=3.0, foreground="white", alpha=0.9), pe.Normal()]
    )

    ax.set_ylim(*safe_ylim_from_series(meas, pred, p90, headroom=HEADROOM_FACTOR))
    ax.set_xlim(TRAIN_START_DT, TEST_END_DT)
    ax.set_xticks(fixed_tick_dates(TRAIN_START_DT, TEST_END_DT, n_ticks=8))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))

    m_train = met_train_nodes[(met_train_nodes.node == node) & (met_train_nodes.split == "TRAIN")].iloc[0]
    m_val   = met_train_nodes[(met_train_nodes.node == node) & (met_train_nodes.split == "VAL")].iloc[0]
    m_test  = met_train_nodes[(met_train_nodes.node == node) & (met_train_nodes.split == "TEST")].iloc[0]

    txt = (
        f"TRAIN  RMSE={m_train.rmse:.2f}  MAE={m_train.mae:.2f}  R²={m_train.r2:.2f}\n"
        f"VAL    RMSE={m_val.rmse:.2f}  MAE={m_val.mae:.2f}  R²={m_val.r2:.2f}\n"
        f"TEST   RMSE={m_test.rmse:.2f}  MAE={m_test.mae:.2f}  R²={m_test.r2:.2f}"
    )
    metric_box(ax, txt, loc="upper right", fontsize=METRIC_FONTSIZE_TS)

    ax.legend(loc="upper center", bbox_to_anchor=(0.20, 0.98), framealpha=0.95, fontsize=10)
    ax.set_title(f"Node {node} — Timeseries (Train/Val/Test)", fontsize=14)
    ax.set_xlabel("Date")
    ax.set_ylabel("mm")
    ax.grid(True, alpha=0.30)

    fig.tight_layout()
    out_png = os.path.join(TS_DIR, f"node_{int(node):02d}_timeseries_full.png")
    fig.savefig(out_png, dpi=SAVE_DPI, bbox_inches="tight")
    print("Saved:", out_png)
    if SHOW_INLINE:
        plt.show()
    plt.close(fig)

for node in sorted(test_nodes):
    j = all_nodes.index(node)
    mask = (dates >= TEST_START_DT) & (dates <= TEST_END_DT)

    tdates = dates[mask]
    meas = Y_full_mm.iloc[:, j].values[mask]
    pred = pred_full_mean.iloc[:, j].values[mask]
    p10  = pred_full_p10.iloc[:, j].values[mask]
    p90  = pred_full_p90.iloc[:, j].values[mask]

    fig, ax = plt.subplots(figsize=(12.0, 6.5))

    ax.plot(
        tdates, pred,
        color=FORECAST_COLOR,
        linewidth=1.8,
        label="Forecast (mean, MC dropout)",
        zorder=4
    )
    ax.fill_between(
        tdates, p10, p90,
        color=BAND_COLOR,
        alpha=BAND_ALPHA,
        label="P10–P90 (MC)",
        zorder=1
    )
    ax.plot(
        tdates, meas,
        color=MEASURED_COLOR,
        linewidth=1.6,
        label="Measured",
        zorder=6,
        path_effects=[pe.Stroke(linewidth=3.0, foreground="white", alpha=0.9), pe.Normal()]
    )

    ax.set_ylim(*safe_ylim_from_series(meas, pred, p90, headroom=HEADROOM_FACTOR))
    ax.set_xlim(TEST_START_DT, TEST_END_DT)
    ax.set_xticks(fixed_tick_dates(TEST_START_DT, TEST_END_DT, n_ticks=8))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))

    m_test = met_test_nodes[(met_test_nodes.node == node) & (met_test_nodes.split == "TEST")].iloc[0]
    txt = f"TEST  RMSE={m_test.rmse:.2f}  MAE={m_test.mae:.2f}  R²={m_test.r2:.2f}"
    metric_box(ax, txt, loc="upper right", fontsize=METRIC_FONTSIZE_TS)

    ax.legend(loc="upper center", bbox_to_anchor=(0.20, 0.98), framealpha=0.95, fontsize=10)
    ax.set_title(f"Node {node} — Timeseries (TEST only)", fontsize=14)
    ax.set_xlabel("Date")
    ax.set_ylabel("mm")
    ax.grid(True, alpha=0.30)

    fig.tight_layout()
    out_png = os.path.join(TS_DIR, f"node_{int(node):02d}_timeseries_test.png")
    fig.savefig(out_png, dpi=SAVE_DPI, bbox_inches="tight")
    print("Saved:", out_png)
    if SHOW_INLINE:
        plt.show()
    plt.close(fig)

# =========================================================
# 15) SCATTER PLOTS
# =========================================================
def scatter_per_split_hex(y_true_mm, y_pred_mm, node, split, out_dir, m_row):
    x = np.log1p(np.clip(y_true_mm, 0, None))
    y = np.log1p(np.clip(y_pred_mm, 0, None))

    finite_mask = np.isfinite(x) & np.isfinite(y)
    x_f = x[finite_mask]
    y_f = y[finite_mask]
    if x_f.size == 0:
        return

    fig, ax = plt.subplots(figsize=(5.0, 4.5))

    hb = ax.hexbin(x_f, y_f, gridsize=60, cmap="viridis", mincnt=1, bins="log")
    cb = fig.colorbar(hb, ax=ax)
    cb.set_label("Count")

    vmin = min(x_f.min(), y_f.min())
    vmax = max(x_f.max(), y_f.max())
    ax.plot([vmin, vmax], [vmin, vmax], linestyle="--", linewidth=1.5, color="#1f77b4", label="y = x")

    if x_f.size > 1:
        a, b = np.polyfit(x_f, y_f, 1)
        xx = np.linspace(vmin, vmax, 100)
        ax.plot(xx, a * xx + b, color="orange", linewidth=2.0, label=f"fit: y={a:.2f}x+{b:.2f}")

    ax.set_xlabel("log1p(Actual mm)")
    ax.set_ylabel("log1p(Predicted mm)")
    ax.set_title(f"Node {node} — Scatter ({split}, log-log)")

    txt = f"RMSE={m_row.rmse:.2f}\nMAE={m_row.mae:.2f}\nR²={m_row.r2:.2f}"
    metric_box(ax, txt, loc="lower right", fontsize=METRIC_FONTSIZE_SC)

    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.30)

    fig.tight_layout()
    out_png = os.path.join(out_dir, f"node_{int(node):02d}_scatter_{split.lower()}.png")
    fig.savefig(out_png, dpi=SAVE_DPI, bbox_inches="tight")
    print("Saved scatter:", out_png)
    if SHOW_INLINE:
        plt.show()
    plt.close(fig)

for node in sorted(train_nodes):
    j = all_nodes.index(node)

    m_train = met_train_nodes[(met_train_nodes.node == node) & (met_train_nodes.split == "TRAIN")].iloc[0]
    scatter_per_split_hex(y_tr_true_mm[:, j], y_tr_mc_mean[:, j], node, "TRAIN", SCAT_DIR, m_train)

    m_val = met_train_nodes[(met_train_nodes.node == node) & (met_train_nodes.split == "VAL")].iloc[0]
    scatter_per_split_hex(y_va_true_mm[:, j], y_va_mc_mean[:, j], node, "VAL", SCAT_DIR, m_val)

    m_test = met_train_nodes[(met_train_nodes.node == node) & (met_train_nodes.split == "TEST")].iloc[0]
    scatter_per_split_hex(y_te_true_mm[:, j], y_te_mc_mean[:, j], node, "TEST", SCAT_DIR, m_test)

for node in sorted(test_nodes):
    j = all_nodes.index(node)
    m_test = met_test_nodes[(met_test_nodes.node == node) & (met_test_nodes.split == "TEST")].iloc[0]
    scatter_per_split_hex(y_te_true_mm[:, j], y_te_mc_mean[:, j], node, "TEST", SCAT_DIR, m_test)

# =========================================================
# 16) ZIP OUTPUTS
# =========================================================
zip_name = os.path.join(OUT_DIR, "lstm_metrics_plots_compromise.zip")
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for fp in [
        METRICS_ALL_CSV,
        TRAIN_METRICS_CSV,
        HELDOUT_METRICS_CSV,
        AGG_TRAIN_CSV,
        AGG_HELDOUT_CSV,
        TRAIN_TABLE_CSV,
        HELDOUT_TABLE_CSV,
        BEST_PARAMS,
        SPLIT_JSON,
    ]:
        if os.path.exists(fp):
            zf.write(fp, os.path.join("analytics", os.path.basename(fp)))

    for fp in [adj_csv, net_png_geo, net_png_circ, BEST_MODEL_PT]:
        if os.path.exists(fp):
            zf.write(fp, os.path.join("network", os.path.basename(fp)))

    for base_dir in [SCAT_DIR, TS_DIR]:
        if os.path.isdir(base_dir):
            for root, _, files in os.walk(base_dir):
                for f in files:
                    if f.lower().endswith(".png"):
                        full = os.path.join(root, f)
                        rel = os.path.relpath(full, PLOTS_DIR)
                        zf.write(full, os.path.join("plots", rel))

print("\nDone.")
print("Plots folder:", PLOTS_DIR)
print("Analytics folder:", AN_DIR)
print("ZIP:", zip_name)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

GCRNN

In [ ]:
# Full corrected GCRNN code
# Purpose:
#   - keep TRAIN and VAL metrics for the 32 training nodes using the model selected on TRAIN→VAL
#   - add TEST (2020–2023) metrics for those same 32 training nodes
#   - keep TEST metrics for the 8 held-out nodes using the final model retrained on TRAIN+VAL
#   - export thesis-ready CSV tables

# ==================================== CUBA — Adjacency + GCRNN (GCN + LSTM) ====================================
# Data splits:
#   TRAIN: 1979-01-01 .. 2015-12-31
#   VAL  : 2016-01-01 .. 2019-12-31
#   TEST : 2020-01-01 .. 2023-12-31
# ===============================================================================================================

import os, glob, json, itertools, csv, random, gc, warnings, secrets
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.dates as mdates

import networkx as nx
from geopy.distance import geodesic

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from dataclasses import dataclass

# ------------------------------- Run/IO config -------------------------------
SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("medium")
    except Exception:
        pass

OUT_DIR   = "gcrnn_cuba_run"
PLOTS_DIR = os.path.join(OUT_DIR, "plots"); os.makedirs(PLOTS_DIR, exist_ok=True)
ALL_PLOTS = os.path.join(PLOTS_DIR, "all_nodes"); os.makedirs(ALL_PLOTS, exist_ok=True)
AN_DIR    = os.path.join(OUT_DIR, "analytics"); os.makedirs(AN_DIR, exist_ok=True)
NET_DIR   = os.path.join(OUT_DIR, "network"); os.makedirs(NET_DIR, exist_ok=True)

SPLIT_JSON= os.path.join(AN_DIR, "node_split.json")
BEST_JSON = os.path.join(AN_DIR, "best.json")
BEST_PT   = os.path.join(AN_DIR, "best.pt")
FINAL_PT  = os.path.join(AN_DIR, "final.pt")
LOG_CSV   = os.path.join(AN_DIR, "tuning_log.csv")

SHOW_INLINE = True
SAVE_DPI    = 180
mpl.rcParams["figure.dpi"] = 140
mpl.rcParams["savefig.dpi"] = 300

# ------------------------------- Global font settings ------------------------
mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 13
mpl.rcParams["xtick.labelsize"] = 11
mpl.rcParams["ytick.labelsize"] = 11
mpl.rcParams["legend.fontsize"] = 11
mpl.rcParams["figure.titlesize"] = 14

# ------------------------------- Re-randomize split? ------------------------
RE_RANDOMIZE_SPLIT = False
SPLIT_SEED = secrets.randbits(32) if RE_RANDOMIZE_SPLIT else SEED
print(f"🎲 Train seed: {SEED}")

# ------------------------------- Date windows -------------------------------
TRAIN_START, TRAIN_END = "1979-01-01", "2015-12-31"
VAL_START,   VAL_END   = "2016-01-01", "2019-12-31"
TEST_START,  TEST_END  = "2020-01-01", "2023-12-31"
CUTOFF_MAX              = "2023-12-31"

# ------------------------------- Fixed timeline windows ---------------------
TRAIN_TS_START = pd.to_datetime("1979-01-01")
TRAIN_TS_END   = pd.to_datetime("2023-12-31")
TEST_TS_START  = pd.to_datetime("2020-01-01")
TEST_TS_END    = pd.to_datetime("2023-12-31")

# ------------------------------- Graph knobs --------------------------------
R2_THR   = 0.60
K_MIN    = 8
ALPHA    = 0.65
L_KM     = 120.0
MIN_W    = 1e-6

# ------------------------------- Model/train knobs --------------------------
LOOKBACK = 60
WET_MM   = 1.0
USE_LOG1P= True

POS_WET   = 10.0
GRAD_CLIP = 0.6
BATCH_SIZE = 64
EPOCHS_SEARCH, ES_PATIENCE_SEARCH = 16, 4
EPOCHS_FINAL,  ES_PATIENCE_FINAL  = 40, 6

TAU_Q_HI, TAU_Q_LO = 0.90, 0.10
AMT_W_GAMMA        = 0.70
AMT_MSE_MIX        = 0.10

RNN_GRID = [
    {"rnn":"LSTM", "layers":1, "hidden":128, "rnn_dropout":0.0},
    {"rnn":"GRU",  "layers":1, "hidden":160, "rnn_dropout":0.0},
]
OPT_GRID = [
    {"lr":1e-3,  "wd":1e-4},
    {"lr":8e-4,  "wd":5e-5},
]
GC_GRID  = [
    {"gc_h1":128, "gc_h2":128, "drop_p":0.15},
    {"gc_h1":160, "gc_h2":160, "drop_p":0.20},
]

# ------------------------------- Fonts for metrics/panels -------------------
METRIC_FONTSIZE_SCATTER = 12
METRIC_FONTSIZE_TS      = 12
PANEL_FONTSIZE          = 16
PANEL_FONTWEIGHT        = "bold"

# ------------------------------- Find / read data ---------------------------
def find_first(patterns):
    for pat in patterns:
        hits = glob.glob(pat, recursive=True)
        if hits:
            return sorted(hits)[0]
    return None

DATA_CSV = find_first([
    "./Cuba_Precipitation_with_Nodes (1).csv",
    "./Cuba_Precipitation_with_Nodes.csv",
    "./cuba_precipitation.csv",
    "/kaggle/input/**/Cuba_Precipitation_with_Nodes*.csv",
    "/kaggle/input/**/cuba_precipitation*.csv",
    "/content/drive/**/Cuba_Precipitation_with_Nodes*.csv",
    "**/*Cuba*Precipitation*Nodes*.csv",
])
assert DATA_CSV, " Precipitation CSV not found. Place it near this script or set DATA_CSV manually."
print("📄 Using data:", DATA_CSV)

raw = pd.read_csv(DATA_CSV)
raw.columns = [c.lower().strip() for c in raw.columns]
needed = {"node","time","precipitation"}
assert needed.issubset(raw.columns), f"CSV must contain {needed}."
raw["time"] = pd.to_datetime(raw["time"], errors="coerce")
raw = raw.dropna(subset=["time"]).copy()
raw = raw.rename(columns={"time":"date", "precipitation":"mm"})
raw["node"] = pd.to_numeric(raw["node"], errors="coerce").astype("Int64")
raw["mm"]   = pd.to_numeric(raw["mm"], errors="coerce").clip(lower=0)
raw = raw[raw["date"] <= pd.to_datetime(CUTOFF_MAX)]

wide0 = raw.pivot(index="date", columns="node", values="mm").sort_index()

have_nodes = [int(n) for n in wide0.columns if wide0[n].fillna(0).sum() > 0]
missing_nodes = sorted(set(range(1,41)) - set(have_nodes))
if missing_nodes:
    print(f"⚠️ Skipping missing/empty nodes: {missing_nodes}")

wide = wide0.reindex(columns=have_nodes).fillna(0.0)
dates = wide.index
all_nodes = list(wide.columns)
print(f"📊 Nodes present: {len(all_nodes)}")

# ------------------------------- Fixed node split ---------------------------
TRAIN_FIXED = [1, 3, 4, 5, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
               20, 21, 23, 24, 27, 28, 29, 30, 31, 32, 33, 34, 35, 37, 38, 40]
TEST_FIXED  = [2, 6, 9, 22, 25, 26, 36, 39]

train_nodes = [n for n in TRAIN_FIXED if n in all_nodes]
test_nodes  = [n for n in TEST_FIXED  if n in all_nodes]

with open(SPLIT_JSON, "w") as f:
    json.dump({"seed": int(SPLIT_SEED), "train_nodes": train_nodes, "test_nodes": test_nodes}, f, indent=2)

print(f" Node split → TRAIN={len(train_nodes)}; TEST={len(test_nodes)}")
print("TRAIN:", train_nodes)
print("TEST :", test_nodes)

# ------------------------------- Helpers ------------------------------------
def rmse(a,b): return float(np.sqrt(mean_squared_error(a,b))) if len(a) else np.nan
def mae(a,b):  return float(mean_absolute_error(a,b)) if len(a) else np.nan

def r2(yt,yp):
    try:
        return float(r2_score(yt, yp))
    except Exception:
        return np.nan

def time_feats(idx: pd.DatetimeIndex) -> np.ndarray:
    m = idx.month.values
    doy = idx.dayofyear.values
    return np.stack([
        np.sin(2*np.pi*m/12.),
        np.cos(2*np.pi*m/12.),
        np.sin(2*np.pi*doy/365.),
        np.cos(2*np.pi*doy/365.)
    ], axis=1).astype(np.float32)

def fixed_tick_dates(start_dt, end_dt, n_ticks=8):
    return pd.date_range(start=start_dt, end=end_dt, periods=n_ticks)

# ------------------------------- Adjacency (TRAIN only) ---------------------
def _maybe_latlon(df: pd.DataFrame):
    lat_col = next((c for c in ["lat","latitude"] if c in df.columns), None)
    lon_col = next((c for c in ["lon","longitude"] if c in df.columns), None)
    if lat_col is None or lon_col is None:
        return None
    g = df.dropna(subset=[lat_col, lon_col, "node"])[["node", lat_col, lon_col]].drop_duplicates("node")
    g["node"] = pd.to_numeric(g["node"], errors="coerce").astype("Int64")
    return g.set_index("node").sort_index().rename(columns={lat_col:"lat", lon_col:"lon"})

def build_adjacency(wide_df, raw_df, train_start, train_end, r2_thr, k_min, alpha, L_km, eps=1e-9):
    cols = [c for c in wide_df.columns]
    mask = (wide_df.index>=pd.to_datetime(train_start)) & (wide_df.index<=pd.to_datetime(train_end))
    rho  = wide_df.loc[mask, cols].corr(method="spearman").astype(float)
    Wcorr = np.square(rho.values)
    Wcorr = np.nan_to_num(Wcorr, nan=0.0)
    Wcorr = 0.5*(Wcorr + Wcorr.T)
    np.fill_diagonal(Wcorr, 0.0)

    geo = _maybe_latlon(raw_df)
    if geo is not None and set(cols).issubset(set(geo.index)):
        lat = geo.loc[cols, "lat"].values.astype(float)
        lon = geo.loc[cols, "lon"].values.astype(float)
        N = len(cols)
        D = np.zeros((N,N), dtype=float)
        for i in range(N):
            a = (lat[i], lon[i])
            D[i,:] = [geodesic(a, (lat[j],lon[j])).km for j in range(N)]
        Kdist = np.exp(-D / max(L_km, 1e-6))
        np.fill_diagonal(Kdist, 0.0)
        Wsoft = Wcorr * Kdist
    else:
        Wsoft = Wcorr

    if r2_thr is not None:
        Wsoft = np.where(Wcorr >= r2_thr, Wsoft, 0.0)

    keep = np.zeros_like(Wsoft, dtype=bool)
    for i in range(Wsoft.shape[0]):
        idx = np.argsort(Wsoft[i])[::-1]
        top = [j for j in idx if j != i][:k_min]
        keep[i, top] = True
    keep = np.logical_or(keep, keep.T)

    A_nb = np.where(keep, Wsoft, 0.0)
    rowsum = A_nb.sum(axis=1, keepdims=True)
    A_row = np.divide(A_nb, rowsum + eps, where=rowsum > 0)
    A_hat = (alpha * np.eye(A_row.shape[0], dtype=float)) + ((1.0 - alpha) * A_row)
    return A_hat.astype(np.float32), Wsoft.astype(np.float32), cols

A_hat, Wsoft, cols_order = build_adjacency(wide, raw, TRAIN_START, TRAIN_END, R2_THR, K_MIN, ALPHA, L_KM)

adj_csv = os.path.join(NET_DIR, f"adj_weighted_r2_{R2_THR:.2f}_L_{int(L_KM)}_train_1979_2015.csv")
pd.DataFrame(Wsoft, index=cols_order, columns=cols_order).to_csv(adj_csv)
print("Saved weighted (pre-normalized) adjacency to:", adj_csv)

# ------------------------------- Network plots ------------------------------
coords = _maybe_latlon(raw)
net_png = os.path.join(NET_DIR, "network_softweights_geo.png")
if coords is not None:
    pos_geo = {n: (coords.loc[n,"lon"], coords.loc[n,"lat"]) for n in cols_order if n in coords.index}
    G = nx.Graph(); G.add_nodes_from(cols_order)
    for i in range(len(cols_order)):
        for j in range(i+1, len(cols_order)):
            w = float(Wsoft[i,j])
            if w >= MIN_W:
                G.add_edge(cols_order[i], cols_order[j], weight=w)
    fig, ax = plt.subplots(figsize=(13, 5.4))
    ax.set_aspect("equal")
    if G.number_of_edges() > 0:
        wvals = np.array([G[u][v]["weight"] for u,v in G.edges()])
        wmin, wmax = float(wvals.min()), float(wvals.max())
        widths = [0.7 + 4.8 * ((G[u][v]["weight"] - wmin) / (wmax - wmin + 1e-12)) for (u,v) in G.edges()]
        nx.draw_networkx_edges(G, pos_geo, width=widths, edge_color="#808080", alpha=0.7, ax=ax)
    nx.draw_networkx_nodes(G, pos_geo, nodelist=[n for n in cols_order if n in train_nodes], node_color="#1e90ff", edgecolors="black", linewidths=1.1, node_size=650, label="Train", ax=ax)
    nx.draw_networkx_nodes(G, pos_geo, nodelist=[n for n in cols_order if n in test_nodes], node_color="#ffa500", edgecolors="black", linewidths=1.1, node_size=650, label="Test", ax=ax)
    texts = nx.draw_networkx_labels(G, pos_geo, labels={n: str(n) for n in cols_order}, font_size=11, font_weight="bold", ax=ax)
    for t in texts.values():
        t.set_path_effects([pe.withStroke(linewidth=2.3, foreground="white")])
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.grid(True, linestyle="--", alpha=0.3); ax.legend(loc="upper right")
    ax.set_title(f"Cuba network — soft weights  (ρ²≥{R2_THR:.2f}, L={int(L_KM)} km, Top-K≥{K_MIN})")
    fig.tight_layout(); fig.savefig(net_png, dpi=SAVE_DPI, bbox_inches="tight")
    if SHOW_INLINE: plt.show()
    plt.close(fig)
else:
    print("No lat/lon columns → geo map skipped.")

circ_png = os.path.join(NET_DIR, "network_circular.png")
G2 = nx.Graph(); G2.add_nodes_from(cols_order)
for i in range(len(cols_order)):
    for j in range(i+1, len(cols_order)):
        w = float(Wsoft[i,j])
        if w >= MIN_W:
            G2.add_edge(cols_order[i], cols_order[j], weight=w)

pos_circ = nx.circular_layout(G2)
fig, ax = plt.subplots(figsize=(9.5, 9.5))
if G2.number_of_edges() > 0:
    wvals = np.array([G2[u][v]["weight"] for u,v in G2.edges()])
    wmin, wmax = float(wvals.min()), float(wvals.max())
    widths = [0.7 + 3.6 * ((G2[u][v]["weight"] - wmin) / (wmax - wmin + 1e-12)) for (u,v) in G2.edges()]
    nx.draw_networkx_edges(G2, pos_circ, width=widths, edge_color="#9e9e9e", alpha=0.6, ax=ax)
nx.draw_networkx_nodes(G2, pos_circ, nodelist=[n for n in cols_order if n in train_nodes], node_color="dodgerblue", node_size=650, edgecolors="black", linewidths=1.2, label="Train")
nx.draw_networkx_nodes(G2, pos_circ, nodelist=[n for n in cols_order if n in test_nodes], node_color="orange", node_size=650, edgecolors="black", linewidths=1.2, label="Test")
nx.draw_networkx_labels(G2, pos_circ, labels={n: str(n) for n in cols_order}, font_size=11, font_weight="bold")
ax.set_title(f"Cuba network — circular layout  (ρ²≥{R2_THR:.2f}, L={int(L_KM)} km, Top-K≥{K_MIN})")
ax.axis("off"); ax.legend(loc="upper right")
fig.tight_layout(); fig.savefig(circ_png, dpi=SAVE_DPI, bbox_inches="tight")
if SHOW_INLINE: plt.show()
plt.close(fig)

# ------------------------------- Dataset ------------------------------------
wet_prev = (wide.shift(1) > WET_MM).astype(np.float32)

class STHurdleDS(Dataset):
    def __init__(self, wide_mm: pd.DataFrame, wet_prev: pd.DataFrame, lookback: int, use_log1p: bool=True):
        self.idx = wide_mm.index
        mm_raw = wide_mm.values.astype(np.float32)
        self.mm_log = np.log1p(mm_raw) if use_log1p else mm_raw
        self.wet_prev = wet_prev.reindex(wide_mm.index).values.astype(np.float32)
        self.seas = time_feats(self.idx)
        self.L = lookback
        self.N = wide_mm.shape[1]

    def __len__(self):
        return len(self.idx) - self.L + 1

    def __getitem__(self, i):
        s = slice(i, i + self.L)
        t = i + self.L - 1
        x1 = self.mm_log[s, :]
        x2 = self.wet_prev[s, :]
        sea = self.seas[s, :]
        sea_b = np.repeat(sea[:, None, :], self.N, axis=1)
        X = np.stack([x1, x2], axis=2)
        X = np.concatenate([X, sea_b], axis=2)
        mm_t = np.expm1(self.mm_log[t, :])
        wet_mask = (mm_t > WET_MM).astype(np.float32)
        y_occ = wet_mask.copy()
        y_amt_log = self.mm_log[t, :]
        return X.astype(np.float32), y_occ.astype(np.float32), y_amt_log.astype(np.float32), wet_mask.astype(np.float32)

ds = STHurdleDS(wide, wet_prev, LOOKBACK, USE_LOG1P)

def seq_mask_ending_on(mask_dates: np.ndarray) -> np.ndarray:
    idx = wide.index
    n_seq = len(ds)
    seq = np.zeros(n_seq, dtype=bool)
    for t in range(LOOKBACK - 1, len(idx)):
        seq[t - (LOOKBACK - 1)] = bool(mask_dates[t])
    return seq

is_train_date = (dates >= pd.to_datetime(TRAIN_START)) & (dates <= pd.to_datetime(TRAIN_END))
is_val_date   = (dates >= pd.to_datetime(VAL_START))   & (dates <= pd.to_datetime(VAL_END))
is_trval_date = (dates >= pd.to_datetime(TRAIN_START)) & (dates <= pd.to_datetime(VAL_END))

idx_tr   = np.where(seq_mask_ending_on(is_train_date))[0]
idx_va   = np.where(seq_mask_ending_on(is_val_date))[0]
idx_trva = np.where(seq_mask_ending_on(is_trval_date))[0]

num_workers = max(0, os.cpu_count() // 2)
dl_tr = DataLoader(ds, batch_size=BATCH_SIZE, sampler=SubsetRandomSampler(idx_tr), drop_last=False, num_workers=num_workers, pin_memory=True, persistent_workers=(num_workers > 0))
dl_va = DataLoader(ds, batch_size=BATCH_SIZE, sampler=SubsetRandomSampler(idx_va), drop_last=False, num_workers=num_workers, pin_memory=True, persistent_workers=(num_workers > 0))
dl_trva = DataLoader(ds, batch_size=BATCH_SIZE, sampler=SubsetRandomSampler(idx_trva), drop_last=False, num_workers=num_workers, pin_memory=True, persistent_workers=(num_workers > 0))

# ------------------------------- Model --------------------------------------
class GraphConv(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        self.lin = nn.Linear(in_f, out_f)

    def forward(self, x, A_hat):
        return self.lin(torch.matmul(A_hat, x))

class STGNN_Hurdle(nn.Module):
    def __init__(self, in_f=6, gc_h1=64, gc_h2=64, rnn_hidden=128, n_layers=1, rnn_type="LSTM", drop=0.15, rnn_dropout=0.0, bidirectional=False):
        super().__init__()
        self.gc1 = GraphConv(in_f, gc_h1)
        self.gc2 = GraphConv(gc_h1, gc_h2)
        self.act = nn.ReLU()
        self.do_gc1 = nn.Dropout(drop)
        self.do_gc2 = nn.Dropout(drop)
        RNN = nn.LSTM if rnn_type.upper() == "LSTM" else nn.GRU
        self.rnn = RNN(
            input_size=gc_h2,
            hidden_size=rnn_hidden,
            num_layers=n_layers,
            dropout=(rnn_dropout if n_layers > 1 else 0.0),
            batch_first=True,
            bidirectional=bidirectional
        )
        out_dim = rnn_hidden * (2 if bidirectional else 1)
        self.do_out = nn.Dropout(drop)
        self.head_occ = nn.Linear(out_dim, 1)
        self.head_amt = nn.Linear(out_dim, 1)
        self.temp = nn.Parameter(torch.ones(1))

    def forward(self, Xseq, A_hat):
        B, L, N, F = Xseq.shape
        X = Xseq.reshape(B * L, N, F)
        h1 = self.do_gc1(self.act(self.gc1(X, A_hat)))
        h2 = self.do_gc2(self.act(self.gc2(h1, A_hat)))
        if h2.shape[-1] == h1.shape[-1]:
            h2 = h2 + h1
        h2 = h2.reshape(B, L, N, -1).permute(0, 2, 1, 3).reshape(B * N, L, -1)
        out, _ = self.rnn(h2)
        hT = self.do_out(out[:, -1, :])
        logit = self.head_occ(hT).reshape(B, N)
        amt   = self.head_amt(hT).reshape(B, N)
        return logit, amt

# ------------------------------- Losses / training --------------------------
bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WET, device=device), reduction="none")
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

def pinball_loss(pred, target, tau=0.5):
    diff = target - pred
    return torch.maximum(tau * diff, (tau - 1) * diff)

def masked_mean(x, mask):
    num = (x * mask).sum()
    den = mask.sum().clamp_min(1.0)
    return num / den

def amount_loss_dual(pred_log, true_log, wet_mask, tau_hi=TAU_Q_HI, tau_lo=TAU_Q_LO, gamma=AMT_W_GAMMA, mse_mix=AMT_MSE_MIX):
    with torch.no_grad():
        mm_true = torch.expm1(true_log).clamp_min(0.0)
        mean_mm = mm_true.mean().clamp_min(1e-6)
        w = (mm_true / mean_mm).pow(gamma)
        w = w / (w.mean().clamp_min(1e-6))
    q_hi = pinball_loss(pred_log, true_log, tau=tau_hi) * w
    q_lo = pinball_loss(pred_log, true_log, tau=tau_lo) * w * 0.35
    mse  = (pred_log - true_log).pow(2) * w * mse_mix
    return masked_mean(q_hi + q_lo + mse, wet_mask)

# ========================= FIXED VALIDATION METRIC =========================
# Validation metric is masked to training nodes only.
@torch.no_grad()
def evaluate_val(model, dl_va, A_torch_, node_mask_vec) -> float:
    model.eval()
    node_mask_t = torch.tensor(node_mask_vec[None, :], dtype=torch.float32, device=device)
    briers = []
    wet_rmses = []
    for X, y_occ, y_amt_log, wet_mask in dl_va:
        X = torch.as_tensor(X, dtype=torch.float32, device=device)
        y_occ = torch.as_tensor(y_occ, dtype=torch.float32, device=device)
        y_amt_log = torch.as_tensor(y_amt_log, dtype=torch.float32, device=device)
        wet_mask = torch.as_tensor(wet_mask, dtype=torch.float32, device=device)

        batch_mask = node_mask_t.repeat(X.size(0), 1)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits, amt = model(X, A_torch_)
            p = torch.sigmoid(logits / model.temp.clamp_min(1e-3))

            # masked Brier score on training nodes only
            brier = masked_mean((p - y_occ) ** 2, batch_mask)
            briers.append(float(brier.item()))

            # masked wet-day RMSE on training nodes only
            pred_mm = torch.expm1(amt).clamp_min(0.0)
            true_mm = torch.expm1(y_amt_log).clamp_min(0.0)
            wet_eval_mask = wet_mask * batch_mask

            if wet_eval_mask.sum() > 0:
                wrmse = torch.sqrt(masked_mean((pred_mm - true_mm) ** 2, wet_eval_mask))
                wet_rmses.append(float(wrmse.item()))
            else:
                wet_rmses.append(0.0)

    return float(np.mean(briers) + np.mean(wet_rmses))

def train_looper(model, dataloader, A_torch_, node_mask_vec, lr, wd, epochs, patience, valid_dl=None):
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2, min_lr=2e-5)
    node_mask_t = torch.tensor(node_mask_vec[None,:], dtype=torch.float32, device=device)
    best_metric = float("inf")
    bad = 0
    best_state = None

    for ep in range(1, epochs + 1):
        model.train(); tr_occ = tr_amt = 0.0; nb = 0
        for X, y_occ, y_amt_log, wet_mask in dataloader:
            X = torch.as_tensor(X, dtype=torch.float32, device=device)
            y_occ = torch.as_tensor(y_occ, dtype=torch.float32, device=device)
            y_amt_log = torch.as_tensor(y_amt_log, dtype=torch.float32, device=device)
            wet_mask = torch.as_tensor(wet_mask, dtype=torch.float32, device=device)

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits, amt = model(X, A_torch_)
                loss_occ_full = bce(logits, y_occ)
                loss_occ = masked_mean(loss_occ_full, node_mask_t.repeat(X.size(0), 1))
                mask_amt = wet_mask * node_mask_t.repeat(X.size(0), 1)
                loss_amt = amount_loss_dual(amt, y_amt_log, mask_amt)
                loss = loss_occ + loss_amt

            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt); scaler.update()

            tr_occ += float(loss_occ.item())
            tr_amt += float(loss_amt.item())
            nb += 1

        val_metric = evaluate_val(model, valid_dl, A_torch_, node_mask_vec) if valid_dl is not None else (tr_occ / nb + tr_amt / nb)
        sched.step(val_metric)
        print(f"[E{ep:02d}] tr_occ={tr_occ/max(1,nb):.4f} tr_amt={tr_amt/max(1,nb):.4f}  val={val_metric:.4f}  lr={opt.param_groups[0]['lr']:.2e}")

        if np.isfinite(val_metric) and (val_metric < best_metric - 1e-4):
            best_metric = val_metric
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print(f"Early stop at epoch {ep}, best={best_metric:.4f}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

# ------------------------------- HP Search ----------------------------------
if not os.path.exists(LOG_CSV):
    with open(LOG_CSV, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["rnn","layers","hidden","rnn_dropout","lr","wd","gc_h1","gc_h2","drop_p","val_metric"])
        w.writeheader()

done = set()
if os.path.exists(LOG_CSV):
    with open(LOG_CSV, "r") as f:
        for row in csv.DictReader(f):
            key = tuple([row[k] for k in ["rnn","layers","hidden","rnn_dropout","lr","wd","gc_h1","gc_h2","drop_p"]])
            done.add(key)

A_torch = torch.tensor(A_hat, dtype=torch.float32, device=device)
node_mask = np.array([1.0 if n in train_nodes else 0.0 for n in all_nodes], dtype=np.float32)

best_score = float("inf")
best_conf  = None
best_model = None
last_model = None
last_conf  = None

for rnn_conf, opt_conf, gc_conf in itertools.product(RNN_GRID, OPT_GRID, GC_GRID):
    conf = {**rnn_conf, **opt_conf, **gc_conf}
    key = tuple(str(conf[k]) for k in ["rnn","layers","hidden","rnn_dropout","lr","wd","gc_h1","gc_h2","drop_p"])
    if key in done:
        print("Skipping", conf)
        continue

    print("\n===== Try:", conf)
    model = STGNN_Hurdle(
        in_f=6,
        gc_h1=conf["gc_h1"],
        gc_h2=conf["gc_h2"],
        rnn_hidden=conf["hidden"],
        n_layers=conf["layers"],
        rnn_type=conf["rnn"],
        drop=conf["drop_p"],
        rnn_dropout=conf["rnn_dropout"],
        bidirectional=False
    ).to(device)

    with torch.no_grad():
        model.temp.fill_(1.0)

    train_looper(model, dl_tr, A_torch, node_mask, lr=conf["lr"], wd=conf["wd"], epochs=EPOCHS_SEARCH, patience=ES_PATIENCE_SEARCH, valid_dl=dl_va)
    score = evaluate_val(model, dl_va, A_torch, node_mask)
    if not np.isfinite(score):
        print("⚠️ val_metric is NaN/inf for config, assigning large penalty.")
        score = 1e6

    print(f"Config {conf} → val_metric={score:.4f}")
    with open(LOG_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["rnn","layers","hidden","rnn_dropout","lr","wd","gc_h1","gc_h2","drop_p","val_metric"])
        w.writerow({**conf, "val_metric": score})

    done.add(key)
    last_model = model
    last_conf  = conf

    if score < best_score:
        best_score = score
        best_conf  = conf
        best_model = model
        torch.save(best_model.state_dict(), BEST_PT)
        with open(BEST_JSON, "w") as f:
            json.dump({"val_metric_best": float(best_score), "best_conf": best_conf}, f)
        print(f"New BEST {best_score:.4f}")

if best_model is None:
    print("All configs produced invalid val_metric. Falling back to last trained configuration.")
    assert last_model is not None, "No models were trained at all."
    best_model = last_model
    best_conf  = last_conf
    torch.save(best_model.state_dict(), BEST_PT)
    with open(BEST_JSON, "w") as f:
        json.dump({"val_metric_best": None, "best_conf": best_conf, "note": "fallback_last_model_due_to_invalid_metrics"}, f)

print("\nBest (or fallback) config used for final retrain:", best_conf)

# ------------------------------- Final retrain -------------------------------
final_model = STGNN_Hurdle(
    in_f=6,
    gc_h1=int(best_conf["gc_h1"]),
    gc_h2=int(best_conf["gc_h2"]),
    rnn_hidden=int(best_conf["hidden"]),
    n_layers=int(best_conf["layers"]),
    rnn_type=str(best_conf["rnn"]),
    drop=float(best_conf["drop_p"]),
    rnn_dropout=float(best_conf["rnn_dropout"]),
    bidirectional=False
).to(device)

final_model.load_state_dict(best_model.state_dict())
with torch.no_grad():
    final_model.temp.copy_(best_model.temp.data)

train_looper(final_model, dl_trva, A_torch, node_mask, lr=float(best_conf["lr"]), wd=float(best_conf["wd"]), epochs=EPOCHS_FINAL, patience=ES_PATIENCE_FINAL, valid_dl=dl_va)
torch.save(final_model.state_dict(), FINAL_PT)

# ------------------------------- Quantile mapping ---------------------------
@dataclass
class QMap:
    x: np.ndarray
    y: np.ndarray
    slope_lo: float
    slope_hi: float
    x_min: float
    x_max: float
    y_min: float
    y_max: float

def _monotone(arr):
    out = np.asarray(arr, float).copy()
    for i in range(1, len(out)):
        if out[i] <= out[i - 1]:
            out[i] = out[i - 1] + 1e-6
    return out

def fit_qmap_from_split(y_true: np.ndarray, y_pred: np.ndarray) -> QMap:
    m = (y_true >= 0) & (y_pred >= 0)
    yt = y_true[m]
    yp = y_pred[m]
    if yt.size < 40:
        xmax = np.log1p(max(1.0, yp.max())) if yp.size else 1.0
        x = np.linspace(0, xmax, 16)
        y = x.copy()
    else:
        q = np.linspace(0.02, 0.98, 61)
        x = np.log1p(np.quantile(yp, q))
        y = np.log1p(np.quantile(yt, q))
        x = _monotone(x); y = _monotone(y)
    dx_lo = max(x[1] - x[0], 1e-4); dy_lo = y[1] - y[0]
    dx_hi = max(x[-1] - x[-2], 1e-4); dy_hi = y[-1] - y[-2]
    slope_lo = float(np.clip(dy_lo / dx_lo, 0.7, 2.5))
    slope_hi = float(np.clip(dy_hi / dx_hi, 0.7, 2.5))
    return QMap(x=x, y=y, slope_lo=slope_lo, slope_hi=slope_hi, x_min=float(x[0]), x_max=float(x[-1]), y_min=float(y[0]), y_max=float(y[-1]))

def apply_qmap(model: QMap, v_pred_mm: np.ndarray) -> np.ndarray:
    xv = np.log1p(np.clip(v_pred_mm, 0, None)).astype(np.float64)
    y = np.interp(xv, model.x, model.y)
    lo = xv < model.x_min
    hi = xv > model.x_max
    y[lo] = model.y_min + model.slope_lo * (xv[lo] - model.x_min)
    y[hi] = model.y_max + model.slope_hi * (xv[hi] - model.x_max)
    return np.expm1(np.clip(y, 0, None))

def fit_bias_models_train_only(train_mask, y_true_df, y_pred_df):
    models = {}
    for n in all_nodes:
        yt = y_true_df.loc[train_mask, n].values
        yp = y_pred_df.loc[train_mask, n].values
        models[n] = fit_qmap_from_split(yt, yp)
    return models

def apply_bias(models, df_pred):
    out = df_pred.copy()
    for n in all_nodes:
        out[n] = apply_qmap(models[n], df_pred[n].values)
    return out

# ------------------------------- Inference ----------------------------------
def build_all_sequences(wide_df: pd.DataFrame, lookback: int, use_log1p=True):
    dts = wide_df.index
    N = wide_df.shape[1]
    mm_raw = wide_df.values.astype(np.float32)
    mm_in = np.log1p(mm_raw) if use_log1p else mm_raw
    wetprv = (wide_df.shift(1) > WET_MM).astype(np.float32).reindex(dts).values
    seas = time_feats(dts)
    seq_idx = []
    X_list = []
    for t in range(lookback - 1, len(dts)):
        s = slice(t - lookback + 1, t + 1)
        x1 = mm_in[s, :]
        x2 = wetprv[s, :]
        sea = seas[s, :]
        sea_b = np.repeat(sea[:, None, :], N, axis=1)
        X = np.stack([x1, x2], axis=2)
        X = np.concatenate([X, sea_b], axis=2).astype(np.float32)
        X_list.append(X)
        seq_idx.append(dts[t])
    X_all = np.stack(X_list, axis=0)
    return X_all, pd.DatetimeIndex(seq_idx)

X_all, seq_dates = build_all_sequences(wide, LOOKBACK, USE_LOG1P)

@torch.no_grad()
def mc_predict(model, A_hat_t, X_all, mc_samples=40, batch=32, tau=TAU_Q_HI):
    S, L, N, F = X_all.shape
    was_training = model.training
    model.train()

    if hasattr(model, "do_gc1") and model.do_gc1.p == 0.0:
        model.do_gc1.p = 0.10
    if hasattr(model, "do_gc2") and model.do_gc2.p == 0.0:
        model.do_gc2.p = 0.10
    if hasattr(model, "do_out") and model.do_out.p == 0.0:
        model.do_out.p = 0.10

    p_samples = []
    a_samples = []
    for _ in range(mc_samples):
        p_out = np.zeros((S, N), dtype=np.float32)
        a_out = np.zeros((S, N), dtype=np.float32)
        for i in range(0, S, batch):
            j = min(i + batch, S)
            X = torch.as_tensor(X_all[i:j], dtype=torch.float32, device=device)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits, amt = model(X, A_hat_t)
                p = torch.sigmoid(logits / model.temp.clamp_min(1e-3))
                a = torch.expm1(amt).clamp_min(0.0)
            p_out[i:j] = p.detach().cpu().numpy()
            a_out[i:j] = a.detach().cpu().numpy()
        p_samples.append(p_out)
        a_samples.append(a_out)

    model.train(was_training)
    p_arr = np.stack(p_samples, axis=0)
    a_arr = np.stack(a_samples, axis=0)
    mm_arr = p_arr * a_arr

    mean_mm = mm_arr.mean(axis=0)
    p10_mm  = np.quantile(mm_arr, 0.10, axis=0)
    p90_mm  = np.quantile(mm_arr, 0.90, axis=0)
    tau_mm  = np.quantile(mm_arr, float(tau), axis=0)

    mean_df = pd.DataFrame(mean_mm, index=seq_dates, columns=all_nodes)
    p10_df  = pd.DataFrame(p10_mm, index=seq_dates, columns=all_nodes)
    p90_df  = pd.DataFrame(p90_mm, index=seq_dates, columns=all_nodes)
    tau_df  = pd.DataFrame(tau_mm, index=seq_dates, columns=all_nodes)
    return mean_df, p10_df, p90_df, tau_df

def infer_and_calibrate(model_for_pred, model_name):
    A_torch_local = torch.tensor(A_hat, dtype=torch.float32, device=device)
    pred_mean_raw, pred_p10_raw, pred_p90_raw, pred_tau_raw = mc_predict(model_for_pred, A_torch_local, X_all, mc_samples=40, batch=32)

    Y_full_raw_local = wide.reindex(seq_dates)
    is_train_idx_local = (seq_dates >= pd.to_datetime(TRAIN_START)) & (seq_dates <= pd.to_datetime(TRAIN_END))
    is_val_idx_local   = (seq_dates >= pd.to_datetime(VAL_START))   & (seq_dates <= pd.to_datetime(VAL_END))
    is_test_idx_local  = (seq_dates >= pd.to_datetime(TEST_START))  & (seq_dates <= pd.to_datetime(TEST_END))

    bias_models_local = fit_bias_models_train_only(is_train_idx_local, Y_full_raw_local, pred_mean_raw)
    pred_mean = apply_bias(bias_models_local, pred_mean_raw)
    pred_p10  = apply_bias(bias_models_local, pred_p10_raw)
    pred_p90  = apply_bias(bias_models_local, pred_p90_raw)
    pred_tau  = apply_bias(bias_models_local, pred_tau_raw)

    return {
        "name": model_name,
        "Y_full_raw": Y_full_raw_local,
        "Pred_mean": pred_mean,
        "Pred_p10": pred_p10,
        "Pred_p90": pred_p90,
        "Pred_tau": pred_tau,
        "is_train_idx": is_train_idx_local,
        "is_val_idx": is_val_idx_local,
        "is_test_idx": is_test_idx_local,
    }

best_model_for_eval = STGNN_Hurdle(
    in_f=6,
    gc_h1=int(best_conf["gc_h1"]),
    gc_h2=int(best_conf["gc_h2"]),
    rnn_hidden=int(best_conf["hidden"]),
    n_layers=int(best_conf["layers"]),
    rnn_type=str(best_conf["rnn"]),
    drop=float(best_conf["drop_p"]),
    rnn_dropout=float(best_conf["rnn_dropout"]),
    bidirectional=False
).to(device)
best_model_for_eval.load_state_dict(torch.load(BEST_PT, map_location=device))
best_model_for_eval.eval()

final_model_for_eval = STGNN_Hurdle(
    in_f=6,
    gc_h1=int(best_conf["gc_h1"]),
    gc_h2=int(best_conf["gc_h2"]),
    rnn_hidden=int(best_conf["hidden"]),
    n_layers=int(best_conf["layers"]),
    rnn_type=str(best_conf["rnn"]),
    drop=float(best_conf["drop_p"]),
    rnn_dropout=float(best_conf["rnn_dropout"]),
    bidirectional=False
).to(device)
final_model_for_eval.load_state_dict(torch.load(FINAL_PT, map_location=device))
final_model_for_eval.eval()

eval_best  = infer_and_calibrate(best_model_for_eval,  "best_model_train_only")
eval_final = infer_and_calibrate(final_model_for_eval, "final_model_train_plus_val")

# ------------------------------- Metrics / exports --------------------------
def split_metrics(node_list, mask_idx, label, y_true_df, y_pred_df, model_used):
    rows = []
    for n in node_list:
        yt = y_true_df.loc[mask_idx, n].values
        yp = y_pred_df.loc[mask_idx, n].values
        rows.append({
            "node": int(n),
            "split": label,
            "rmse": rmse(yt, yp),
            "mae": mae(yt, yp),
            "r2": r2(yt, yp),
            "model_used": model_used,
        })
    return rows

rows_train_nodes = []
rows_train_nodes += split_metrics(train_nodes, eval_best["is_train_idx"], "TRAIN", eval_best["Y_full_raw"], eval_best["Pred_mean"], eval_best["name"])
rows_train_nodes += split_metrics(train_nodes, eval_best["is_val_idx"],   "VAL",   eval_best["Y_full_raw"], eval_best["Pred_mean"], eval_best["name"])
rows_train_nodes += split_metrics(train_nodes, eval_best["is_test_idx"],  "TEST",  eval_best["Y_full_raw"], eval_best["Pred_mean"], eval_best["name"])
met_train_nodes = pd.DataFrame(rows_train_nodes).sort_values(["node", "split"])
met_train_nodes.to_csv(os.path.join(AN_DIR, "per_node_metrics_train_nodes_train_val_test.csv"), index=False)

rows_test_nodes = []
rows_test_nodes += split_metrics(test_nodes, eval_final["is_test_idx"], "TEST", eval_final["Y_full_raw"], eval_final["Pred_mean"], eval_final["name"])
met_test_nodes = pd.DataFrame(rows_test_nodes).sort_values(["node", "split"])
met_test_nodes.to_csv(os.path.join(AN_DIR, "per_node_metrics_heldout_test_nodes_test_only.csv"), index=False)

def agg_report(df, label):
    part = df[df.split == label]
    return {
        "split": label,
        "rmse_mean": float(np.nanmean(part.rmse)),
        "mae_mean": float(np.nanmean(part.mae)),
        "r2_mean": float(np.nanmean(part.r2)),
        "n_nodes": int(part.node.nunique()),
    }

overall_train_nodes = pd.DataFrame([
    agg_report(met_train_nodes, "TRAIN"),
    agg_report(met_train_nodes, "VAL"),
    agg_report(met_train_nodes, "TEST"),
])
overall_train_nodes.to_csv(os.path.join(AN_DIR, "split_aggregates_train_nodes.csv"), index=False)

overall_test_nodes = pd.DataFrame([agg_report(met_test_nodes, "TEST")])
overall_test_nodes.to_csv(os.path.join(AN_DIR, "split_aggregates_heldout_test_nodes.csv"), index=False)

print("\n=== 32 training nodes: TRAIN / VAL / TEST (best_model trained on TRAIN only) ===")
print(overall_train_nodes)
print("\n=== 8 held-out nodes: TEST only (final_model trained on TRAIN+VAL) ===")
print(overall_test_nodes)

train_nodes_table = met_train_nodes.pivot(index="node", columns="split", values=["rmse", "mae", "r2"]).sort_index()
train_nodes_table.columns = [f"{metric.upper()}_{split}" for metric, split in train_nodes_table.columns]
train_nodes_table = train_nodes_table.reset_index()
train_nodes_table.to_csv(os.path.join(AN_DIR, "table_train_nodes_train_val_test.csv"), index=False)

heldout_test_table = met_test_nodes[["node", "rmse", "mae", "r2"]].rename(columns={"rmse": "RMSE_TEST", "mae": "MAE_TEST", "r2": "R2_TEST"}).sort_values("node")
heldout_test_table.to_csv(os.path.join(AN_DIR, "table_heldout_nodes_test_only.csv"), index=False)

print("\nSaved thesis tables:")
print(os.path.join(AN_DIR, "table_train_nodes_train_val_test.csv"))
print(os.path.join(AN_DIR, "table_heldout_nodes_test_only.csv"))

# ------------------------------- Plot helpers -------------------------------
SCATTER_GRIDSIZE = 45
SCATTER_MINCNT   = 1
SCATTER_ALPHA    = 0.95
MIN_MM_FOR_PLOT  = 0.0

def _metrics_text_single(label, yt, yp):
    return f"{label}  RMSE={rmse(yt,yp):.2f}  MAE={mae(yt,yp):.2f}  R²={r2(yt,yp):.2f}"

def scatter_loglog_hex(yt, yp, title, out_png, panel_label=None):
    yt = np.asarray(yt, float); yp = np.asarray(yp, float)
    m = np.isfinite(yt) & np.isfinite(yp)
    yt = yt[m]; yp = yp[m]
    if MIN_MM_FOR_PLOT > 0:
        m2 = (yt >= MIN_MM_FOR_PLOT) | (yp >= MIN_MM_FOR_PLOT)
        yt, yp = yt[m2], yp[m2]

    yt_l = np.log1p(np.clip(yt, 0, None))
    yp_l = np.log1p(np.clip(yp, 0, None))
    lim = float(max(yt_l.max(initial=1e-6), yp_l.max(initial=1e-6), 1e-6))

    fig, ax = plt.subplots(figsize=(5.0, 4.5))
    hb = ax.hexbin(yt_l, yp_l, gridsize=SCATTER_GRIDSIZE, mincnt=SCATTER_MINCNT, linewidths=0, cmap="viridis", bins="log")
    ax.plot([0, lim], [0, lim], '--', label="y = x")

    if yt_l.size >= 3:
        k, b = np.polyfit(yt_l, yp_l, 1)
        xs = np.linspace(0, lim, 100)
        ax.plot(xs, k*xs + b, '-', alpha=0.95, label=f"fit: y={k:.2f}x{b:+.2f}")

    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_aspect('equal', 'box')
    ax.set_xlabel("log1p(Actual mm)", fontsize=13)
    ax.set_ylabel("log1p(Predicted mm)", fontsize=13)
    ax.set_title(title, fontsize=14)
    ax.tick_params(axis="both", labelsize=11)
    ax.legend(loc="upper left", framealpha=SCATTER_ALPHA, fontsize=11)
    ax.grid(True, alpha=0.3)

    txt = "\n".join([f"RMSE={rmse(yt, yp):.2f}", f"MAE={mae(yt, yp):.2f}", f"R²={r2(yt, yp):.2f}"])
    ax.text(0.98, 0.02, txt, transform=ax.transAxes, ha="right", va="bottom", fontsize=METRIC_FONTSIZE_SCATTER, bbox=dict(boxstyle="round", facecolor="white", alpha=0.9))

    if panel_label is not None:
        ax.text(0.02, 0.98, f"({panel_label})", transform=ax.transAxes, ha="left", va="top", fontsize=PANEL_FONTSIZE, fontweight=PANEL_FONTWEIGHT)

    cb = fig.colorbar(hb, ax=ax, pad=0.02)
    cb.set_label("Count", fontsize=12)
    cb.ax.tick_params(labelsize=11)

    fig.tight_layout(); fig.savefig(out_png, dpi=SAVE_DPI, bbox_inches="tight")
    if SHOW_INLINE: plt.show()
    plt.close(fig)

letters_train = list("abcdefghijklmnop")
train_panel_nodes = sorted(train_nodes)[:8]
panel_labels_train = {}
k = 0
for node in train_panel_nodes:
    if k < len(letters_train):
        panel_labels_train[("TRAIN", node)] = letters_train[k]; k += 1
    if k < len(letters_train):
        panel_labels_train[("VAL", node)] = letters_train[k]; k += 1

letters_test = list("qrstuvwx")
panel_labels_test = {node: letters_test[i] for i, node in enumerate(sorted(test_nodes)[:8])}

# ------------------------------- Timeline settings --------------------------
SAFE_CAP_MIN, SAFE_CAP_MAX, SAFE_CAP_FACTOR = 40.0, 1000.0, 3.0
BAND_MIN_MM, MEAN_MIN_GAP_MM = 0.5, 0.2
CLR_FORE, CLR_MEAS, CLR_MEAN, CLR_BAND = "#1f77b4", "#ff7f0e", "#2ca02c", "#9370db"
BAND_ALPHA = 0.28
HEADROOM_FACTOR = 1.35

for n in all_nodes:
    try:
        # scatters
        if n in train_nodes:
            yt = eval_best["Y_full_raw"].loc[eval_best["is_train_idx"], n].values
            yp = eval_best["Pred_mean"].loc[eval_best["is_train_idx"], n].values
            scatter_loglog_hex(
                yt, yp,
                f"Node {n} — Scatter (TRAIN, log-log)",
                os.path.join(ALL_PLOTS, f"node_{n:02d}_scatter_train_loglog.png"),
                panel_label=panel_labels_train.get(("TRAIN", n), None)
            )

            yt = eval_best["Y_full_raw"].loc[eval_best["is_val_idx"], n].values
            yp = eval_best["Pred_mean"].loc[eval_best["is_val_idx"], n].values
            scatter_loglog_hex(
                yt, yp,
                f"Node {n} — Scatter (VAL, log-log)",
                os.path.join(ALL_PLOTS, f"node_{n:02d}_scatter_val_loglog.png"),
                panel_label=panel_labels_train.get(("VAL", n), None)
            )

            # TEST scatter for training nodes
            yt = eval_best["Y_full_raw"].loc[eval_best["is_test_idx"], n].values
            yp = eval_best["Pred_mean"].loc[eval_best["is_test_idx"], n].values
            scatter_loglog_hex(
                yt, yp,
                f"Node {n} — Scatter (TEST, log-log)",
                os.path.join(ALL_PLOTS, f"node_{n:02d}_scatter_test_loglog.png")
            )

        if n in test_nodes:
            yt = eval_final["Y_full_raw"].loc[eval_final["is_test_idx"], n].values
            yp = eval_final["Pred_mean"].loc[eval_final["is_test_idx"], n].values
            scatter_loglog_hex(
                yt, yp,
                f"Node {n} — Scatter (TEST, log-log)",
                os.path.join(ALL_PLOTS, f"node_{n:02d}_scatter_test_loglog.png"),
                panel_label=panel_labels_test.get(n, None)
            )

        # timelines
        if n in test_nodes:
            mask_dates = (seq_dates >= TEST_TS_START) & (seq_dates <= TEST_TS_END)
            x_start, x_end = TEST_TS_START, TEST_TS_END
            title_span = "TEST only"
            xticks = fixed_tick_dates(TEST_TS_START, TEST_TS_END, n_ticks=8)
            actual = eval_final["Y_full_raw"][n].values
            mean_s = eval_final["Pred_mean"][n].values
            tau_s  = eval_final["Pred_tau"][n].values
            p10_s  = eval_final["Pred_p10"][n].values
            p90_s  = eval_final["Pred_p90"][n].values
        else:
            mask_dates = (seq_dates >= TRAIN_TS_START) & (seq_dates <= TRAIN_TS_END)
            x_start, x_end = TRAIN_TS_START, TRAIN_TS_END
            title_span = "Train/Val/Test"
            xticks = fixed_tick_dates(TRAIN_TS_START, TRAIN_TS_END, n_ticks=8)
            actual = eval_best["Y_full_raw"][n].values
            mean_s = eval_best["Pred_mean"][n].values
            tau_s  = eval_best["Pred_tau"][n].values
            p10_s  = eval_best["Pred_p10"][n].values
            p90_s  = eval_best["Pred_p90"][n].values

        if np.any(mask_dates):
            obs_cap  = float(np.nanpercentile(actual[mask_dates], 99.7))
            pred_cap = float(np.nanpercentile(np.maximum.reduce([tau_s[mask_dates], p90_s[mask_dates], mean_s[mask_dates]]), 99.7))
        else:
            obs_cap, pred_cap = 1.0, 1.0

        safe_cap = np.clip(max(SAFE_CAP_MIN, SAFE_CAP_FACTOR * max(obs_cap, pred_cap)), SAFE_CAP_MIN, SAFE_CAP_MAX)
        y_upper = safe_cap * HEADROOM_FACTOR

        dts = seq_dates[mask_dates]
        actual_plot = np.clip(actual[mask_dates], 0, safe_cap)
        mean_plot   = np.clip(mean_s[mask_dates], 0, safe_cap)
        tau_plot    = np.clip(tau_s[mask_dates], 0, safe_cap)
        p10_plot    = np.clip(p10_s[mask_dates], 0, safe_cap)
        p90_plot    = np.clip(p90_s[mask_dates], 0, safe_cap)

        mean_tau_gap = float(np.median(np.abs(mean_plot - tau_plot))) if len(mean_plot) else 0.0
        show_band = True
        show_mean = (mean_tau_gap >= MEAN_MIN_GAP_MM)

        fig, ax = plt.subplots(figsize=(12, 6.5))
        ax.set_xlim(x_start, x_end)
        ax.set_ylim(0, y_upper)
        ax.set_xticks(xticks)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))

        ax.plot(dts, tau_plot, label=f"Forecast (τ={TAU_Q_HI:.2f}, calibrated)", lw=1.8, linestyle="-", color=CLR_FORE, alpha=0.85, zorder=4)
        if show_mean:
            ax.plot(dts, mean_plot, label="Forecast (mean, calibrated)", lw=1.25, linestyle="--", color=CLR_MEAN, alpha=0.9, zorder=3)
        if show_band:
            ax.fill_between(dts, p10_plot, p90_plot, alpha=BAND_ALPHA, color=CLR_BAND, edgecolor="none", label="P10–P90 (MC, calibrated)", zorder=1)
        ax.plot(dts, actual_plot, label="Measured", lw=1.6, color=CLR_MEAS, zorder=6, path_effects=[pe.Stroke(linewidth=3.0, foreground="white", alpha=0.9), pe.Normal()])

        metric_lines = []
        if n in train_nodes:
            metric_lines.append(_metrics_text_single("TRAIN", eval_best["Y_full_raw"].loc[eval_best["is_train_idx"], n].values, eval_best["Pred_mean"].loc[eval_best["is_train_idx"], n].values))
            metric_lines.append(_metrics_text_single("VAL",   eval_best["Y_full_raw"].loc[eval_best["is_val_idx"], n].values,   eval_best["Pred_mean"].loc[eval_best["is_val_idx"], n].values))
            metric_lines.append(_metrics_text_single("TEST",  eval_best["Y_full_raw"].loc[eval_best["is_test_idx"], n].values,  eval_best["Pred_mean"].loc[eval_best["is_test_idx"], n].values))
        if n in test_nodes:
            metric_lines.append(_metrics_text_single("TEST", eval_final["Y_full_raw"].loc[eval_final["is_test_idx"], n].values, eval_final["Pred_mean"].loc[eval_final["is_test_idx"], n].values))

        if metric_lines:
            ax.text(0.99, 0.99, "\n".join(metric_lines), transform=ax.transAxes, ha="right", va="top", fontsize=METRIC_FONTSIZE_TS, bbox=dict(boxstyle="round", facecolor="white", alpha=0.85), zorder=10)

        ax.legend(loc="upper center", bbox_to_anchor=(0.20, 0.98), framealpha=0.95, fontsize=10)
        ax.set_title(f"Node {n} — Timeseries ({title_span}) — thr={R2_THR:.2f}, τ={TAU_Q_HI:.2f} — Calibrated", fontsize=14)
        ax.set_xlabel("Date", fontsize=13)
        ax.set_ylabel("mm", fontsize=13)
        ax.tick_params(axis="both", labelsize=11)
        ax.grid(True, alpha=0.3)
        plt.setp(ax.get_xticklabels(), rotation=0, ha="center")

        fig.tight_layout()
        outp = os.path.join(ALL_PLOTS, f"node_{n:02d}_timeline.png")
        fig.savefig(outp, dpi=SAVE_DPI, bbox_inches="tight")
        if SHOW_INLINE: plt.show()
        plt.close(fig)

    except Exception as e:
        print(f"⚠️ Plotting failed for node {n}: {e}")

print(f"✅ Cuba geo map  → {net_png}")
print(f"✅ Circular net  → {circ_png}")
print(f"✅ Adjacency CSV → {adj_csv}")
print(f"✅ Plots saved   → {ALL_PLOTS}")
print(f"✅ Metrics/Summary CSVs → {AN_DIR}")
gc.collect()

In [ ]:
# =========================================================
# EXPORT / ZIP ALL GCRNN OUTPUTS
# =========================================================
import os
import zipfile

# main output zip
zip_name = os.path.join(OUT_DIR, "gcrnn_metrics_plots_export.zip")

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    # analytics / metrics files
    for fp in [
        SPLIT_JSON,
        BEST_JSON,
        BEST_PT,
        FINAL_PT,
        LOG_CSV,
        os.path.join(AN_DIR, "per_node_metrics_train_nodes_train_val_test.csv"),
        os.path.join(AN_DIR, "per_node_metrics_heldout_test_nodes_test_only.csv"),
        os.path.join(AN_DIR, "split_aggregates_train_nodes.csv"),
        os.path.join(AN_DIR, "split_aggregates_heldout_test_nodes.csv"),
        os.path.join(AN_DIR, "table_train_nodes_train_val_test.csv"),
        os.path.join(AN_DIR, "table_heldout_nodes_test_only.csv"),
    ]:
        if os.path.exists(fp):
            zf.write(fp, os.path.join("analytics", os.path.basename(fp)))

    # network files
    for fp in [
        adj_csv,
        os.path.join(NET_DIR, "network_softweights_geo.png"),
        os.path.join(NET_DIR, "network_circular.png"),
    ]:
        if os.path.exists(fp):
            zf.write(fp, os.path.join("network", os.path.basename(fp)))

    # all plots
    if os.path.isdir(ALL_PLOTS):
        for root, _, files in os.walk(ALL_PLOTS):
            for f in files:
                if f.lower().endswith(".png"):
                    full = os.path.join(root, f)
                    rel = os.path.relpath(full, PLOTS_DIR)
                    zf.write(full, os.path.join("plots", rel))

print("ZIP created:", zip_name)
print("Full path:", os.path.abspath(zip_name))